# Ambient collocation in 50D

**Paper:** *Gaeta–Lie Neural SDEs: Symmetry-Regularized Learning of Stochastic Dynamics* (NeurIPS 2026) — Section 5.3, Table 2, Appendix B.1.

Re-runs Stage III on Ex 50.1–50.3 with 60 training trajectories. All branches share the Stage-I checkpoint and frozen generators; only the composition of the 256-point symmetry batch changes (trajectory / jittered / uniform-box points). Also reports the standardized nearest-neighbor distance diagnostic. Run once per system and seed.

**Pipeline naming used in the code:** Stage I = baseline neural SDE surrogate; Stage II = symmetry discovery over the basis library; Stage F = Stage III (symmetry-regularized refinement).

Run the cells top to bottom. All configuration lives in `CONFIG` in the first code cell. See the [repository README](../../README.md) for installation and the full experiment index.

## Cell 1 — Config, imports, and collocation-protocol settings

In [ ]:
# =============================================================================
# Cell 1 — Config, imports, and collocation-protocol settings
# =============================================================================

from jax import config as jax_config
jax_config.update("jax_enable_x64", False)

import json
import math
import time
from functools import partial

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())


CONFIG = {
    # -------------------------------------------------------------------------
    # Reproducibility
    # -------------------------------------------------------------------------
    # Run the full notebook separately for at least 3 seeds, ideally 5.
    "SEED": 42,

    # Evaluation points must be independent of all training keys.
    "E1_EVAL_SEED": 20260725,
    "E1_REFERENCE_SEED": 20260726,

    # -------------------------------------------------------------------------
    # 50D synthetic SDE generation
    # -------------------------------------------------------------------------
    "STATE_DIM": 50,

    # Manuscript 50D systems:
    #   Ex 50.1: "ring_double_well_50d"
    #   Ex 50.2: "low_rank_ou_50d"
    #   Ex 50.3: "tanh_chain_50d"
    #
    # "block_lorenz_50d" may remain in the registry for optional diagnostics,
    # but it is not part of the primary E1 result.
    "TRUE_SDE_NAME": "tanh_chain_50d",

    "T_FINAL": 2.0,
    "DT": 0.01,

    # The manuscript defines the data-scarce 50D regime as d+10=60.
    "N_TRAJECTORIES": 60,

    "X0_MEAN": 0.0,
    "X0_STD": 1.0,
    "TRAIN_FRACTION": 0.8,

    # -------------------------------------------------------------------------
    # Neural SDE surrogate
    # Input : (t, x1, ..., x50), dimension 51
    # Output: (f1,...,f50, raw_sigma1,...,raw_sigma50), dimension 100
    # -------------------------------------------------------------------------
    "MLP_HIDDEN_DIM": 192,
    "MLP_DEPTH": 3,
    "INPUT_T_SCALE": 2.0,
    "INPUT_X_SCALE": 3.0,
    "SIGMA_EPS": 1e-4,

    # -------------------------------------------------------------------------
    # Stage I: baseline surrogate training
    # -------------------------------------------------------------------------
    "BASELINE_EPOCHS": 2000,
    "BASELINE_BATCH_SIZE": 512,
    "BASELINE_LR": 2e-3,
    "PRINT_EVERY": 250,

    # -------------------------------------------------------------------------
    # Stage II: symmetry discovery
    # IMPORTANT: E1 keeps this stage on the submitted jitter protocol.
    # -------------------------------------------------------------------------
    "N_GENERATORS": 50,
    "SYM_DISCOVERY_EPOCHS": 4000,
    "SYM_DISCOVERY_LR": 5e-3,
    "SYM_DISCOVERY_POINTS": 256,
    "SYM_DISCOVERY_T_BOX": (0.0, 2.0),
    "SYM_DISCOVERY_X_BOX": (-3.0, 3.0),
    "SYM_INIT_SCALE": 0.03,
    "E1_JITTER_STD": 0.05,

    # Stage-II loss weights
    "LAMBDA_DET": 1.0,
    "LAMBDA_CLOSE": 0.5,
    "LAMBDA_IND": 0.05,
    "LAMBDA_PROJECTABLE": 1.0,
    "LAMBDA_NORM_HINGE": 0.01,
    "MIN_GENERATOR_NORM": 0.10,
    "CLOSURE_RCOND": 1e-5,

    # -------------------------------------------------------------------------
    # 50D editable basis
    # -------------------------------------------------------------------------
    "BASIS_MODE": "expanded_gaeta_lowrank",
    "BASIS_LINEAR_MU_LIST": [-1.0, -0.5, 0.5, 1.0],
    "BASIS_EXP_CLIP": 6.0,
    "BASIS_LOW_RANK_K": 6,
    "BASIS_ADD_EXPONENTIAL": True,
    "BASIS_ADD_LOW_RANK_ROTATIONS": True,
    "BASIS_ADD_LOW_RANK_SHEARS": True,

    # -------------------------------------------------------------------------
    # Stage III: E1 paired refinement experiment
    # -------------------------------------------------------------------------
    "STAGE_F_EPOCHS": 800,
    "STAGE_F_BATCH_SIZE": 512,
    "STAGE_F_LR": 3e-4,
    "STAGE_F_LAMBDA_SYM_DET": 105000.0,
    "STAGE_F_LAMBDA_PROJECTABLE": 1e-3,
    "STAGE_F_EXTRA_T_BOX": (0.0, 2.0),
    "STAGE_F_EXTRA_X_BOX": (-3.0, 3.0),

    # Every symmetry-trained branch receives this exact total.
    "E1_TOTAL_SYM_POINTS": 256,

    # Optional support-aware ambient branch.
    # Leave False for the primary controlled E1 comparison. Turn on only after
    # the core branches run correctly and if additional compute is available.
    "E1_ENABLE_SUPPORT_AMBIENT": False,
    "E1_SUPPORT_Q_LOW": 0.01,
    "E1_SUPPORT_Q_HIGH": 0.99,

    # -------------------------------------------------------------------------
    # E1 evaluation budgets
    # -------------------------------------------------------------------------
    "E1_COEFF_EVAL_POINTS": 2000,
    "E1_RESIDUAL_EVAL_POINTS": 256,
    "E1_RESIDUAL_CHUNK_SIZE": 16,

    "E1_DISTANCE_REFERENCE_POINTS": 4096,
    "E1_DISTANCE_PROBE_POINTS": 512,
    "E1_DISTANCE_CHUNK_SIZE": 64,

    # Single-run directional-check thresholds. These are diagnostics, not
    # substitutes for a multi-seed confidence interval.
    "E1_OFF_MANIFOLD_FRACTION_THRESHOLD": 0.80,
    "E1_NLL_REL_TOL": 0.05,

    # Optional plots retained outside the final numerical reporting cell.
    "N_PLOT_GRID": 160,
    "PLOT_COMPONENTS": [0, 1, 2, 9, 24, 49],
}


D = CONFIG["STATE_DIM"]
Q = D + 1


E1_EXAMPLE_LABELS = {
    "ring_double_well_50d": "Ex 50.1",
    "low_rank_ou_50d": "Ex 50.2",
    "tanh_chain_50d": "Ex 50.3",
    "block_lorenz_50d": "optional non-manuscript diagnostic",
}


# All symmetry branches sum to E1_TOTAL_SYM_POINTS=256.
# The NLL-only branch intentionally contains no symmetry points.
E1_BRANCH_SPECS = {
    "nll_continuation": {
        "label": "NLL-only continuation",
        "use_symmetry": False,
        "n_traj": 0,
        "n_jitter": 0,
        "n_box": 0,
        "n_support": 0,
    },
    "sym_traj_only": {
        "label": "trajectory-only symmetry",
        "use_symmetry": True,
        "n_traj": 256,
        "n_jitter": 0,
        "n_box": 0,
        "n_support": 0,
    },
    "sym_traj_plus_jitter": {
        "label": "submitted trajectory+jitter",
        "use_symmetry": True,
        "n_traj": 128,
        "n_jitter": 128,
        "n_box": 0,
        "n_support": 0,
    },
    "sym_traj_plus_box": {
        "label": "trajectory+uniform-box ambient",
        "use_symmetry": True,
        "n_traj": 128,
        "n_jitter": 0,
        "n_box": 128,
        "n_support": 0,
    },
    "sym_box_only": {
        "label": "pure uniform-box ambient",
        "use_symmetry": True,
        "n_traj": 0,
        "n_jitter": 0,
        "n_box": 256,
        "n_support": 0,
    },
    "sym_traj_jitter_box": {
        "label": "trajectory+jitter+uniform-box",
        "use_symmetry": True,
        "n_traj": 128,
        "n_jitter": 64,
        "n_box": 64,
        "n_support": 0,
    },
}

if CONFIG["E1_ENABLE_SUPPORT_AMBIENT"]:
    E1_BRANCH_SPECS["sym_traj_plus_support_ambient"] = {
        "label": "trajectory+support-aware ambient",
        "use_symmetry": True,
        "n_traj": 128,
        "n_jitter": 0,
        "n_box": 0,
        "n_support": 128,
    }


E1_MODEL_LABELS = {
    "stage1_baseline": "Stage-I baseline",
    **{name: spec["label"] for name, spec in E1_BRANCH_SPECS.items()},
}


key = jax.random.PRNGKey(CONFIG["SEED"])

print("\nE1 experiment configuration")
print("  SDE       :", CONFIG["TRUE_SDE_NAME"])
print("  example   :", E1_EXAMPLE_LABELS.get(CONFIG["TRUE_SDE_NAME"], "unknown"))
print("  seed      :", CONFIG["SEED"])
print("  trajectories:", CONFIG["N_TRAJECTORIES"])
print("  E1 branches:")
for name, spec in E1_BRANCH_SPECS.items():
    print(
        f"    {name:30s} | symmetry={spec['use_symmetry']} | "
        f"traj={spec['n_traj']:3d} | jitter={spec['n_jitter']:3d} | "
        f"box={spec['n_box']:3d} | support={spec['n_support']:3d}"
    )


## Cell 2 — Editable 50D SDE registry and synthetic data

In [ ]:
# ============================================================
# Cell 2 — Editable 50D SDE registry and synthetic data
# ============================================================

# ------------------------------------------------------------------
# How to use:
#
# Change CONFIG["TRUE_SDE_NAME"] in Cell 1.
#
# Available examples:
#   "ring_double_well_50d"
#   "low_rank_ou_50d"
#   "block_lorenz_50d"
#   "tanh_chain_50d"
#
# Each SDE entry defines:
#   drift_fn(t, x) -> shape (..., 50)
#   sigma_fn(t, x) -> shape (..., 50), positive diagonal diffusion
# ------------------------------------------------------------------


def _idx_float():
    return jnp.arange(D, dtype=jnp.float32)


def _phase_vector():
    return 2.0 * jnp.pi * _idx_float() / float(D)


def _mode_sin(k):
    return jnp.sin(2.0 * jnp.pi * k * _idx_float() / float(D))


def _mode_cos(k):
    return jnp.cos(2.0 * jnp.pi * k * _idx_float() / float(D))


def _normalize_mode(v):
    return v / (jnp.linalg.norm(v) + 1e-8)


# ============================================================
# 50D SDE example 1: ring-coupled double-well
# ============================================================

def drift_ring_double_well_50d(t, x):
    """
    50D nonlinear ring-coupled double-well SDE.

    Drift:
        f_i = x_i - x_i^3
              + c (x_{i-1} + x_{i+1} - 2 x_i)
              + weak time forcing_i(t)

    Nontrivial features:
        - nonlinear bistability in every coordinate
        - local spatial coupling on a ring
        - weak heterogeneous forcing
    """
    c = 0.25
    lap = jnp.roll(x, 1, axis=-1) + jnp.roll(x, -1, axis=-1) - 2.0 * x

    phase = _phase_vector()
    forcing = 0.08 * jnp.sin(2.0 * jnp.pi * t + phase)

    return x - x**3 + c * lap + forcing


def sigma_ring_double_well_50d(t, x):
    """
    Positive state-dependent diagonal diffusion.
    """
    phase = _phase_vector()
    base = 0.18 + 0.02 * jnp.sin(phase) ** 2
    return base + 0.04 * jax.nn.sigmoid(x)


# ============================================================
# 50D SDE example 2: low-rank coupled OU/spiral system
# ============================================================

def drift_low_rank_ou_50d(t, x):
    """
    Stable 50D linear/nonlinear low-rank OU-like SDE.

    Drift:
        stable coordinate damping
        + skew ring coupling
        + low-rank global coupling
        + weak periodic forcing
        + small tanh nonlinearity

    This is useful because the true drift has global low-rank structure
    but is still learned by a generic neural SDE.
    """
    idx = _idx_float()

    # Coordinate-dependent damping
    lam = 0.45 + 0.30 * (idx / float(D - 1))

    # Ring skew/transport-like coupling
    skew = 0.18 * (jnp.roll(x, -1, axis=-1) - jnp.roll(x, 1, axis=-1))

    v1 = _normalize_mode(_mode_sin(1))
    v2 = _normalize_mode(_mode_cos(2))
    v3 = _normalize_mode(_mode_sin(3))

    p1 = jnp.sum(x * v1, axis=-1, keepdims=True)
    p2 = jnp.sum(x * v2, axis=-1, keepdims=True)
    p3 = jnp.sum(x * v3, axis=-1, keepdims=True)

    low_rank = (
        0.45 * p1 * v2
        - 0.35 * p2 * v1
        + 0.25 * jnp.tanh(p3) * v3
    )

    forcing = 0.05 * (
        jnp.sin(1.2 * t) * v1
        + jnp.cos(0.8 * t) * v2
    )

    return -lam * x + skew + low_rank + forcing + 0.05 * jnp.tanh(x)


def sigma_low_rank_ou_50d(t, x):
    v1 = _normalize_mode(_mode_sin(1))
    p1 = jnp.sum(x * v1, axis=-1, keepdims=True)
    base = 0.20 + 0.03 * jnp.sin(_phase_vector()) ** 2
    return base + 0.03 * jax.nn.sigmoid(p1) + 0.02 * jnp.tanh(x) ** 2


# ============================================================
# 50D SDE example 3: block Lorenz-style system
# ============================================================

def drift_block_lorenz_50d(t, x):
    """
    50D block-tamed Lorenz-style SDE.

    First 48 coordinates form 16 weakly coupled Lorenz-like triples.
    Last 2 coordinates are stable auxiliary variables coupled to block means.

    The Lorenz blocks are tamed to avoid numerical explosions.
    """
    x_blocks = x[..., :48].reshape(x.shape[:-1] + (16, 3))
    y = x[..., 48:50]

    x1 = x_blocks[..., 0]
    x2 = x_blocks[..., 1]
    x3 = x_blocks[..., 2]

    sigma_l = 4.0
    rho = 8.0
    beta = 8.0 / 3.0

    g1 = sigma_l * (x2 - x1)
    g2 = x1 * (rho - x3) - x2
    g3 = x1 * x2 - beta * x3

    raw = jnp.stack([g1, g2, g3], axis=-1)

    norm2 = jnp.sum(x_blocks**2, axis=-1, keepdims=True)
    tame = 1.0 + 0.05 * norm2

    block_f = raw / tame

    # Weak coupling between neighboring blocks
    block_lap = (
        jnp.roll(x_blocks, 1, axis=-2)
        + jnp.roll(x_blocks, -1, axis=-2)
        - 2.0 * x_blocks
    )
    block_f = block_f + 0.05 * block_lap

    # Auxiliary coordinates
    block_mean = jnp.mean(x_blocks, axis=(-2, -1), keepdims=False)
    f_y1 = -0.60 * y[..., 0] + 0.10 * jnp.tanh(block_mean)
    f_y2 = -0.75 * y[..., 1] + 0.10 * jnp.sin(t)

    f_y = jnp.stack([f_y1, f_y2], axis=-1)

    return jnp.concatenate([block_f.reshape(x.shape[:-1] + (48,)), f_y], axis=-1)


def sigma_block_lorenz_50d(t, x):
    base = 0.12 + 0.02 * jnp.sin(_phase_vector()) ** 2
    return base + 0.03 * jnp.tanh(x) ** 2


# ============================================================
# 50D SDE example 4: nonlinear tanh reaction chain
# ============================================================

def drift_tanh_chain_50d(t, x):
    """
    50D smooth nonlinear reaction-chain SDE.

    Drift:
        damping
        + forward/backward tanh couplings
        + weak global low-rank feedback
        + heterogeneous forcing

    This is bounded and stable, but still nonlinear and coupled.
    """
    phase = _phase_vector()

    f_local = (
        -0.55 * x
        + 0.35 * jnp.tanh(jnp.roll(x, -1, axis=-1))
        - 0.25 * jnp.tanh(jnp.roll(x, 1, axis=-1))
        + 0.10 * jnp.tanh(jnp.roll(x, -2, axis=-1))
    )

    v1 = _normalize_mode(_mode_cos(1))
    v2 = _normalize_mode(_mode_sin(2))

    p1 = jnp.sum(x * v1, axis=-1, keepdims=True)
    p2 = jnp.sum(x * v2, axis=-1, keepdims=True)

    global_feedback = 0.20 * jnp.tanh(p1) * v2 - 0.15 * jnp.tanh(p2) * v1

    forcing = 0.04 * jnp.sin(1.5 * t + phase)

    return f_local + global_feedback + forcing


def sigma_tanh_chain_50d(t, x):
    return 0.22 + 0.04 * jnp.sin(x) ** 2


# ============================================================
# Registry
# ============================================================

SDE_EXAMPLES = {
    "ring_double_well_50d": {
        "description": "50D nonlinear ring-coupled double-well SDE with state-dependent diffusion.",
        "drift_fn": drift_ring_double_well_50d,
        "sigma_fn": sigma_ring_double_well_50d,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 0.75,
        "input_x_scale": 2.5,
        "x_box": (-2.5, 2.5),
    },

    "low_rank_ou_50d": {
        "description": "50D stable low-rank OU/spiral SDE with global modes and state-dependent diffusion.",
        "drift_fn": drift_low_rank_ou_50d,
        "sigma_fn": sigma_low_rank_ou_50d,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 1.0,
        "input_x_scale": 3.0,
        "x_box": (-3.0, 3.0),
    },

    "block_lorenz_50d": {
        "description": "50D tamed Lorenz-block SDE: 16 coupled triples plus 2 stable auxiliary coordinates.",
        "drift_fn": drift_block_lorenz_50d,
        "sigma_fn": sigma_block_lorenz_50d,
        "t_final": 1.0,
        "dt": 0.005,
        "x0_mean": 0.0,
        "x0_std": 0.8,
        "input_x_scale": 5.0,
        "x_box": (-5.0, 8.0),
    },

    "tanh_chain_50d": {
        "description": "50D damped nonlinear tanh reaction chain with low-rank feedback.",
        "drift_fn": drift_tanh_chain_50d,
        "sigma_fn": sigma_tanh_chain_50d,
        "t_final": 3.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 1.0,
        "input_x_scale": 3.0,
        "x_box": (-3.0, 3.0),
    },
}


def activate_sde_from_config():
    """
    Select active SDE from CONFIG["TRUE_SDE_NAME"] and update
    dependent CONFIG fields automatically.
    """
    name = CONFIG["TRUE_SDE_NAME"]

    if name not in SDE_EXAMPLES:
        raise ValueError(
            f"Unknown TRUE_SDE_NAME={name}. "
            f"Available choices: {list(SDE_EXAMPLES.keys())}"
        )

    spec = SDE_EXAMPLES[name]

    CONFIG["T_FINAL"] = float(spec["t_final"])
    CONFIG["DT"] = float(spec["dt"])
    CONFIG["X0_MEAN"] = spec["x0_mean"]
    CONFIG["X0_STD"] = spec["x0_std"]
    CONFIG["INPUT_T_SCALE"] = float(spec["t_final"])
    CONFIG["INPUT_X_SCALE"] = float(spec["input_x_scale"])

    x_box = tuple(float(v) for v in spec["x_box"])

    CONFIG["SYM_DISCOVERY_T_BOX"] = (0.0, float(spec["t_final"]))
    CONFIG["STAGE_F_EXTRA_T_BOX"] = (0.0, float(spec["t_final"]))

    CONFIG["SYM_DISCOVERY_X_BOX"] = x_box
    CONFIG["STAGE_F_EXTRA_X_BOX"] = x_box

    return spec


ACTIVE_SDE = activate_sde_from_config()


def _as_50d_vector(value, name):
    """
    Convert scalar or length-D array/tuple into shape (D,).
    """
    arr = jnp.asarray(value, dtype=jnp.float32)

    if arr.ndim == 0:
        return jnp.full((D,), arr, dtype=jnp.float32)

    if arr.shape != (D,):
        raise ValueError(f"{name} must be scalar or shape ({D},), got {arr.shape}")

    return arr


def true_drift(t, x):
    """
    Dispatch to active 50D drift.

    x shape:
        (..., 50)

    return shape:
        (..., 50)
    """
    return ACTIVE_SDE["drift_fn"](t, x)


def true_sigma(t, x):
    """
    Dispatch to active 50D diagonal diffusion.

    x shape:
        (..., 50)

    return shape:
        (..., 50)
    """
    sigma = ACTIVE_SDE["sigma_fn"](t, x)
    return jnp.maximum(sigma, 1e-6)


def print_active_sde_report():
    print("\n" + "=" * 72)
    print("ACTIVE 50D SYNTHETIC SDE")
    print("=" * 72)
    print(f"TRUE_SDE_NAME: {CONFIG['TRUE_SDE_NAME']}")
    print(f"Description  : {ACTIVE_SDE['description']}")
    print(f"STATE_DIM    : {D}")
    print(f"T_FINAL      : {CONFIG['T_FINAL']}")
    print(f"DT           : {CONFIG['DT']}")
    print(f"X0_MEAN      : {CONFIG['X0_MEAN']}")
    print(f"X0_STD       : {CONFIG['X0_STD']}")
    print(f"X_BOX        : {CONFIG['STAGE_F_EXTRA_X_BOX']}")
    print(f"INPUT_T_SCALE: {CONFIG['INPUT_T_SCALE']}")
    print(f"INPUT_X_SCALE: {CONFIG['INPUT_X_SCALE']}")
    print("=" * 72)


print_active_sde_report()


@jax.jit
def simulate_sde_paths(key):
    """
    Simulate all trajectories using Euler-Maruyama.

    Returns
    -------
    times : shape (n_steps + 1,)
    paths : shape (n_paths, n_steps + 1, 50)
    """
    T = CONFIG["T_FINAL"]
    dt = CONFIG["DT"]
    n_steps = int(round(T / dt))
    n_paths = CONFIG["N_TRAJECTORIES"]

    key_x0, key_noise = jax.random.split(key)

    x0_mean = _as_50d_vector(CONFIG["X0_MEAN"], "X0_MEAN")
    x0_std = _as_50d_vector(CONFIG["X0_STD"], "X0_STD")

    x0 = x0_mean[None, :] + x0_std[None, :] * jax.random.normal(
        key_x0,
        (n_paths, D),
    )

    noise = jax.random.normal(key_noise, (n_steps, n_paths, D))

    times = jnp.arange(n_steps + 1, dtype=jnp.float32) * dt
    sqrt_dt = jnp.sqrt(dt)

    def step(x, inp):
        t, zeta = inp
        f = true_drift(t, x)
        s = true_sigma(t, x)
        x_next = x + f * dt + s * sqrt_dt * zeta
        return x_next, x_next

    _, xs_tail = jax.lax.scan(step, x0, (times[:-1], noise))

    paths = jnp.concatenate([x0[None, :, :], xs_tail], axis=0)
    paths = jnp.transpose(paths, (1, 0, 2))

    return times, paths


def make_increment_dataset(times, paths):
    """
    Convert simulated trajectories into one-step increment data.

    Returns:
        t0 : shape (n_samples,)
        x0 : shape (n_samples, 50)
        dt : shape (n_samples,)
        dx : shape (n_samples, 50)
    """
    n_paths, n_times, dim = paths.shape
    n_steps = n_times - 1
    dt = CONFIG["DT"]

    t0 = jnp.broadcast_to(times[:-1][None, :], (n_paths, n_steps)).reshape(-1)
    x0 = paths[:, :-1, :].reshape(-1, dim)
    dx = (paths[:, 1:, :] - paths[:, :-1, :]).reshape(-1, dim)
    dt_arr = jnp.full((x0.shape[0],), dt, dtype=jnp.float32)

    return t0, x0, dt_arr, dx


def train_test_split_dataset(key, dataset, train_fraction):
    n = dataset[0].shape[0]
    perm = jax.random.permutation(key, n)
    n_train = int(train_fraction * n)

    idx_train = perm[:n_train]
    idx_test = perm[n_train:]

    train = tuple(arr[idx_train] for arr in dataset)
    test = tuple(arr[idx_test] for arr in dataset)

    return train, test


def sample_batch(key, dataset, batch_size):
    n = dataset[0].shape[0]
    idx = jax.random.randint(key, (batch_size,), 0, n)
    return tuple(arr[idx] for arr in dataset)


# ----------------------------
# Generate data
# ----------------------------
key, subkey = jax.random.split(key)
times, paths = simulate_sde_paths(subkey)
dataset = make_increment_dataset(times, paths)

key, subkey = jax.random.split(key)
train_data, test_data = train_test_split_dataset(
    subkey,
    dataset,
    CONFIG["TRAIN_FRACTION"],
)

print("times shape:", times.shape)
print("paths shape:", paths.shape)
print("train samples:", train_data[0].shape[0])
print("test samples:", test_data[0].shape[0])


# ----------------------------
# Quick trajectory plots for selected components only
# ----------------------------
for comp in CONFIG["PLOT_COMPONENTS"]:
    plt.figure(figsize=(8, 4))
    for i in range(min(12, CONFIG["N_TRAJECTORIES"])):
        plt.plot(np.array(times), np.array(paths[i, :, comp]), alpha=0.6)
    plt.xlabel("t")
    plt.ylabel(f"X_{comp+1}(t)")
    plt.title(
        f"{CONFIG['TRUE_SDE_NAME']} trajectories: component {comp+1}"
    )
    plt.grid(True)
    plt.show()


# ----------------------------
# Sanity check drift/diffusion values
# ----------------------------
key, subkey = jax.random.split(key)
x_probe = jax.random.normal(subkey, (8, D))
t_probe = 0.5 * CONFIG["T_FINAL"]

print("\nDrift probe shape:", true_drift(t_probe, x_probe).shape)
print("Sigma probe shape:", true_sigma(t_probe, x_probe).shape)
print("Sigma probe min  :", float(jnp.min(true_sigma(t_probe, x_probe))))

# =============================================================================
# ADDITION TO CELL 2
# Keep all existing 50D SDE definitions and data-generation code.
# Insert this block immediately AFTER train_data and test_data are created and
# their sample counts are printed.
# =============================================================================

E1_EXAMPLE_LABEL = E1_EXAMPLE_LABELS.get(
    CONFIG["TRUE_SDE_NAME"],
    "unknown 50D example",
)

e1_train_x = train_data[1]
e1_n_train_states = int(e1_train_x.shape[0])

e1_stats_std = jnp.std(e1_train_x, axis=0)
E1_TRAIN_X_MEAN = jnp.mean(e1_train_x, axis=0)
E1_TRAIN_X_STD = jnp.maximum(e1_stats_std, 1e-6)
E1_SUPPORT_X_LOW = jnp.quantile(
    e1_train_x,
    CONFIG["E1_SUPPORT_Q_LOW"],
    axis=0,
)
E1_SUPPORT_X_HIGH = jnp.quantile(
    e1_train_x,
    CONFIG["E1_SUPPORT_Q_HIGH"],
    axis=0,
)

# Construct disjoint reference and on-data probe sets. The probe set is not
# allowed to contain its own nearest-neighbour reference, preventing zero
# distances by construction.
e1_ref_key = jax.random.fold_in(
    jax.random.PRNGKey(CONFIG["E1_REFERENCE_SEED"]),
    CONFIG["SEED"],
)
e1_ref_perm = jax.random.permutation(e1_ref_key, e1_n_train_states)

e1_n_ref = min(
    CONFIG["E1_DISTANCE_REFERENCE_POINTS"],
    max(1, e1_n_train_states // 2),
)
e1_n_probe = min(
    CONFIG["E1_DISTANCE_PROBE_POINTS"],
    e1_n_train_states - e1_n_ref,
)

if e1_n_probe <= 0:
    raise ValueError(
        "Not enough training states to form disjoint E1 reference/probe sets."
    )

E1_DISTANCE_REFERENCE_X = e1_train_x[e1_ref_perm[:e1_n_ref]]
E1_DISTANCE_ONDATA_PROBE_X = e1_train_x[
    e1_ref_perm[e1_n_ref:e1_n_ref + e1_n_probe]
]

print("\nE1 state-support preparation")
print("  manuscript example          :", E1_EXAMPLE_LABEL)
print("  training states             :", e1_n_train_states)
print("  distance reference states   :", E1_DISTANCE_REFERENCE_X.shape[0])
print("  disjoint on-data probe states:", E1_DISTANCE_ONDATA_PROBE_X.shape[0])
print(
    "  training std range          :",
    float(jnp.min(E1_TRAIN_X_STD)),
    "to",
    float(jnp.max(E1_TRAIN_X_STD)),
)
print(
    "  support-aware lower range   :",
    float(jnp.min(E1_SUPPORT_X_LOW)),
    "to",
    float(jnp.max(E1_SUPPORT_X_LOW)),
)
print(
    "  support-aware upper range   :",
    float(jnp.min(E1_SUPPORT_X_HIGH)),
    "to",
    float(jnp.max(E1_SUPPORT_X_HIGH)),
)
# =============================================================================
# EARLY E1 OFF-MANIFOLD GEOMETRY DIAGNOSTIC
# Insert immediately before Cell 3.
# =============================================================================

def e1_early_standardized_nn_distances(query_x):
    """
    Nearest-neighbour distance in standardized 50D state space:

        d_NN(x) = min_r sqrt(sum_j ((x_j-r_j)/train_std_j)^2)

    Time is intentionally excluded.
    """
    query_x = jnp.asarray(query_x, dtype=jnp.float32)
    reference_x = E1_DISTANCE_REFERENCE_X
    scale = E1_TRAIN_X_STD

    distance_pieces = []
    chunk_size = int(CONFIG["E1_DISTANCE_CHUNK_SIZE"])

    for start in range(0, int(query_x.shape[0]), chunk_size):
        stop = min(start + chunk_size, int(query_x.shape[0]))
        query_chunk = query_x[start:stop]

        standardized_difference = (
            query_chunk[:, None, :] - reference_x[None, :, :]
        ) / scale[None, None, :]

        squared_distance = jnp.sum(
            standardized_difference**2,
            axis=-1,
        )

        nearest_distance = jnp.sqrt(
            jnp.min(squared_distance, axis=1)
        )

        distance_pieces.append(np.asarray(nearest_distance))

    return np.concatenate(distance_pieces, axis=0)


# Use exactly the same deterministic evaluation keys as the final cell.
e1_early_eval_root = jax.random.PRNGKey(CONFIG["E1_EVAL_SEED"])
e1_early_eval_keys = jax.random.split(e1_early_eval_root, 16)

n_geometry_points = int(CONFIG["E1_DISTANCE_PROBE_POINTS"])


# -------------------------------------------------------------------------
# 1. Disjoint on-data probe states
# -------------------------------------------------------------------------
e1_early_ondata_x = E1_DISTANCE_ONDATA_PROBE_X


# -------------------------------------------------------------------------
# 2. Near-trajectory jitter states
# Replicates sample_tx_data_jitter(eval_keys[6], ...).
# -------------------------------------------------------------------------
e1_jitter_idx_key, e1_jitter_noise_key = jax.random.split(
    e1_early_eval_keys[6]
)

e1_jitter_indices = jax.random.randint(
    e1_jitter_idx_key,
    (n_geometry_points,),
    0,
    train_data[1].shape[0],
)

e1_early_jitter_x = (
    train_data[1][e1_jitter_indices]
    + CONFIG["E1_JITTER_STD"]
    * jax.random.normal(
        e1_jitter_noise_key,
        (n_geometry_points, D),
    )
)


# -------------------------------------------------------------------------
# 3. Uniform-box states
# Replicates the state component of sample_tx_box(eval_keys[7], ...).
# -------------------------------------------------------------------------
_, e1_box_x_key = jax.random.split(e1_early_eval_keys[7])

e1_box_low, e1_box_high = CONFIG["STAGE_F_EXTRA_X_BOX"]
e1_box_low = jnp.asarray(e1_box_low, dtype=jnp.float32)
e1_box_high = jnp.asarray(e1_box_high, dtype=jnp.float32)

if e1_box_low.ndim == 0:
    e1_box_low = jnp.full((D,), e1_box_low, dtype=jnp.float32)

if e1_box_high.ndim == 0:
    e1_box_high = jnp.full((D,), e1_box_high, dtype=jnp.float32)

e1_box_uniform = jax.random.uniform(
    e1_box_x_key,
    (n_geometry_points, D),
)

e1_early_box_x = (
    e1_box_low[None, :]
    + e1_box_uniform
    * (e1_box_high - e1_box_low)[None, :]
)


# -------------------------------------------------------------------------
# Compute standardized nearest-neighbour distances
# -------------------------------------------------------------------------
e1_early_ondata_distances = e1_early_standardized_nn_distances(
    e1_early_ondata_x
)

e1_early_jitter_distances = e1_early_standardized_nn_distances(
    e1_early_jitter_x
)

e1_early_box_distances = e1_early_standardized_nn_distances(
    e1_early_box_x
)


# -------------------------------------------------------------------------
# Requested summary quantities
# -------------------------------------------------------------------------
e1_early_ondata_mean = float(
    np.mean(e1_early_ondata_distances)
)

e1_early_ondata_p95 = float(
    np.quantile(e1_early_ondata_distances, 0.95)
)

e1_early_jitter_mean = float(
    np.mean(e1_early_jitter_distances)
)

e1_early_box_mean = float(
    np.mean(e1_early_box_distances)
)

e1_early_box_ondata_ratio = (
    e1_early_box_mean / max(e1_early_ondata_mean, 1e-12)
)

e1_early_box_beyond_ondata_p95 = float(
    np.mean(
        e1_early_box_distances > e1_early_ondata_p95
    )
)


# -------------------------------------------------------------------------
# Print before any neural-network training begins
# -------------------------------------------------------------------------
print("\n" + "=" * 78)
print("EARLY E1 OFF-MANIFOLD GEOMETRY DIAGNOSTIC")
print("=" * 78)
print(f"SDE                               : {CONFIG['TRUE_SDE_NAME']}")
print(f"Example                           : {E1_EXAMPLE_LABEL}")
print(f"Number of probe points            : {n_geometry_points}")
print(f"On-data mean d_NN                 : {e1_early_ondata_mean:.6f}")
print(f"On-data p95 d_NN                  : {e1_early_ondata_p95:.6f}")
print(f"Jitter mean d_NN                  : {e1_early_jitter_mean:.6f}")
print(f"Uniform-box mean d_NN             : {e1_early_box_mean:.6f}")
print(f"Box/on-data mean-distance ratio   : {e1_early_box_ondata_ratio:.4f}x")
print(
    "Box points beyond on-data p95    : "
    f"{e1_early_box_beyond_ondata_p95:.6f} "
    f"({100.0 * e1_early_box_beyond_ondata_p95:.2f}%)"
)
print("=" * 78)


## Cell 3 — 50D Neural SDE surrogate, Adam optimizer, baseline training

In [ ]:
# ============================================================
# Cell 3 — 50D Neural SDE surrogate, Adam optimizer, baseline training
# ============================================================

def glorot_uniform(key, in_dim, out_dim):
    lim = jnp.sqrt(6.0 / (in_dim + out_dim))
    return jax.random.uniform(key, (in_dim, out_dim), minval=-lim, maxval=lim)


class MLP:
    """
    Small pure-JAX MLP class.

    For the 3D SDE:
        input  = (t, x1, x2, x3), dimension 4
        output = (f1, f2, f3, raw_sigma1, raw_sigma2, raw_sigma3), dimension 6
    """

    def __init__(self, in_dim=4, hidden_dim=96, depth=3, out_dim=6):
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim
        self.depth = depth
        self.out_dim = out_dim

    def init(self, key):
        sizes = [self.in_dim] + [self.hidden_dim] * self.depth + [self.out_dim]
        keys = jax.random.split(key, len(sizes) - 1)

        params = []
        for k, in_d, out_d in zip(keys, sizes[:-1], sizes[1:]):
            W = glorot_uniform(k, in_d, out_d)
            b = jnp.zeros((out_d,), dtype=jnp.float32)
            params.append({"W": W, "b": b})
        return params

    def apply(self, params, z):
        h = z
        for layer in params[:-1]:
            h = jnp.tanh(h @ layer["W"] + layer["b"])
        out = h @ params[-1]["W"] + params[-1]["b"]
        return out


surrogate_model = MLP(
    in_dim=Q,
    hidden_dim=CONFIG["MLP_HIDDEN_DIM"],
    depth=CONFIG["MLP_DEPTH"],
    out_dim=2 * D,
)


def normalize_input(t, x):
    tn = jnp.asarray([t / CONFIG["INPUT_T_SCALE"]], dtype=jnp.float32)
    xn = x / CONFIG["INPUT_X_SCALE"]
    return jnp.concatenate([tn, xn], axis=0)


def surrogate_f_sigma(params, t, x):
    z = normalize_input(t, x)
    raw = surrogate_model.apply(params, z)

    f = raw[:D]
    sigma = jax.nn.softplus(raw[D:]) + CONFIG["SIGMA_EPS"]
    return f, sigma


def f_vec_single(params, z):
    """
    z = (t, x1, x2, x3)
    returns f(t,x), shape (3,)
    """
    return surrogate_f_sigma(params, z[0], z[1:])[0]


def sigma_vec_single(params, z):
    """
    z = (t, x1, x2, x3)
    returns diagonal sigma(t,x), shape (3,)
    """
    return surrogate_f_sigma(params, z[0], z[1:])[1]


def data_nll_loss(params, batch):
    """
    Euler-Maruyama diagonal-Gaussian negative log-likelihood.

    batch = (t0, x0, dt, dx)
        t0 shape (B,)
        x0 shape (B, 3)
        dt shape (B,)
        dx shape (B, 3)
    """
    t0, x0, dt, dx = batch

    f, sigma = jax.vmap(lambda tt, xx: surrogate_f_sigma(params, tt, xx))(t0, x0)

    mean = f * dt[:, None]
    var = sigma**2 * dt[:, None] + 1e-8

    nll_components = 0.5 * (
        (dx - mean) ** 2 / var
        + jnp.log(2.0 * jnp.pi * var)
    )

    return jnp.mean(jnp.sum(nll_components, axis=-1))


# ----------------------------
# Lightweight pure-JAX Adam
# ----------------------------
def adam_init(params):
    m = jax.tree_util.tree_map(jnp.zeros_like, params)
    v = jax.tree_util.tree_map(jnp.zeros_like, params)
    t = jnp.array(0, dtype=jnp.int32)
    return {"m": m, "v": v, "t": t}


def adam_update(params, grads, state, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    t_new = state["t"] + 1

    m_new = jax.tree_util.tree_map(
        lambda m, g: beta1 * m + (1.0 - beta1) * g,
        state["m"],
        grads,
    )
    v_new = jax.tree_util.tree_map(
        lambda v, g: beta2 * v + (1.0 - beta2) * (g * g),
        state["v"],
        grads,
    )

    m_hat = jax.tree_util.tree_map(lambda m: m / (1.0 - beta1**t_new), m_new)
    v_hat = jax.tree_util.tree_map(lambda v: v / (1.0 - beta2**t_new), v_new)

    params_new = jax.tree_util.tree_map(
        lambda p, m, v: p - lr * m / (jnp.sqrt(v) + eps),
        params,
        m_hat,
        v_hat,
    )

    return params_new, {"m": m_new, "v": v_new, "t": t_new}


@jax.jit
def baseline_train_step(params, opt_state, batch):
    loss, grads = jax.value_and_grad(data_nll_loss)(params, batch)
    params, opt_state = adam_update(
        params,
        grads,
        opt_state,
        CONFIG["BASELINE_LR"],
    )
    return params, opt_state, loss


# ----------------------------
# Train baseline surrogate
# ----------------------------
key, subkey = jax.random.split(key)
sde_params = surrogate_model.init(subkey)
sde_opt_state = adam_init(sde_params)

baseline_losses = []
t_start = time.time()

for epoch in range(1, CONFIG["BASELINE_EPOCHS"] + 1):
    key, subkey = jax.random.split(key)
    batch = sample_batch(subkey, train_data, CONFIG["BASELINE_BATCH_SIZE"])

    sde_params, sde_opt_state, loss = baseline_train_step(
        sde_params,
        sde_opt_state,
        batch,
    )

    baseline_losses.append(float(loss))

    if epoch % CONFIG["PRINT_EVERY"] == 0 or epoch == 1:
        train_loss = float(data_nll_loss(sde_params, train_data))
        test_loss = float(data_nll_loss(sde_params, test_data))
        print(
            f"[Baseline] epoch {epoch:5d} | "
            f"batch nll {float(loss): .5f} | "
            f"train nll {train_loss: .5f} | "
            f"test nll {test_loss: .5f}"
        )

print(f"Baseline training time: {time.time() - t_start:.2f} sec")

# =============================================================================
# ADDITION TO CELL 3
# Keep the current model, optimizer, data loss, and Stage-I training code.
# Add this block immediately AFTER Stage-I training finishes.
#
# Also change the stale Cell-3 comments/docstrings from "3D" to "50D".
# =============================================================================

def tree_copy(tree):
    """Create an explicit immutable copy of a JAX pytree."""
    return jax.tree_util.tree_map(lambda x: jnp.array(x), tree)


def tree_block_until_ready(tree):
    """Synchronize every JAX array in a pytree."""
    leaves = jax.tree_util.tree_leaves(tree)
    for leaf in leaves:
        if hasattr(leaf, "block_until_ready"):
            leaf.block_until_ready()
    return tree


# This is the one common initialization for every E1 Stage-III branch.
sde_params_baseline = tree_copy(sde_params)
tree_block_until_ready(sde_params_baseline)

baseline_train_nll_at_stage1 = float(
    data_nll_loss(sde_params_baseline, train_data)
)
baseline_test_nll_at_stage1 = float(
    data_nll_loss(sde_params_baseline, test_data)
)

print("\nFrozen Stage-I checkpoint for E1")
print(f"  train NLL: {baseline_train_nll_at_stage1: .8f}")
print(f"  test  NLL: {baseline_test_nll_at_stage1: .8f}")

## Cell 4 — Editable 50D Gaeta-style expanded symmetry-basis library

In [ ]:
# ============================================================
# Cell 4 — Editable 50D Gaeta-style expanded symmetry-basis library
# ============================================================

# ------------------------------------------------------------------
# Each basis returns a 51-vector:
#     [tau, xi1, ..., xi50]
#
# Minimal 104-basis:
#   3 time-only generators:
#       ∂t, t∂t, t^2∂t
#   50 coordinate translations:
#       ∂x_i
#   50 coordinate scalings:
#       x_i ∂x_i
#   1 global Brownian/Gaeta scaling:
#       2(t/T)∂t + x·∇x
#
# Expanded Gaeta/low-rank additions:
#   1. Exponential translations:
#       e^{mu t/T} ∂x_i
#
#   2. Exponential projective linear-drift generators:
#       e^{2mu t/T}(∂t + mu x·∇x)
#
#   3. Low-rank rotations:
#       (v_b·x)v_a·∇x - (v_a·x)v_b·∇x
#
#   4. Low-rank shears:
#       (v_b·x)v_a·∇x
#
# The added basis elements are all at most linear in x, so xi_xx = 0.
# Therefore the Hessian-free determining-equation implementation remains valid.
# ------------------------------------------------------------------

BASIS_LIBRARY = []


def add_basis(name, source, fn):
    BASIS_LIBRARY.append({"name": name, "source": source, "fn": fn})


def time_basis(t):
    """
    Normalized time variable.

    We use t/T rather than raw t inside exponentials and polynomial
    time basis functions for numerical stability.
    """
    return t / CONFIG["INPUT_T_SCALE"]


def safe_exp(z):
    c = CONFIG["BASIS_EXP_CLIP"]
    return jnp.exp(jnp.clip(z, -c, c))


def zero_xi():
    return jnp.zeros((D,), dtype=jnp.float32)


def e_i(i):
    v = jnp.zeros((D,), dtype=jnp.float32)
    return v.at[i].set(1.0)


def unit(v):
    return v / (jnp.linalg.norm(v) + 1e-8)


def make_low_rank_directions(K):
    """
    Generic low-rank directions in coordinate-index space.

    These are not fitted from the data and do not use the true SDE.
    They are fixed Fourier-like directions:
        constant, sin(1), cos(1), sin(2), cos(2), ...
    """
    idx = jnp.arange(D, dtype=jnp.float32)
    dirs = []

    # Global mean direction.
    dirs.append(unit(jnp.ones((D,), dtype=jnp.float32)))

    freq = 1
    while len(dirs) < K:
        phase = 2.0 * jnp.pi * freq * (idx + 0.5) / float(D)

        dirs.append(unit(jnp.sin(phase)))

        if len(dirs) < K:
            dirs.append(unit(jnp.cos(phase)))

        freq += 1

    return tuple(dirs[:K])


# ============================================================
# Group A: time-only projectable basis
# ============================================================

add_basis(
    "time_translation__dt",
    "Generic projectable time translation",
    lambda t, x: jnp.concatenate(
        [jnp.array([1.0], dtype=jnp.float32), zero_xi()]
    ),
)

add_basis(
    "time_scaling__t_dt",
    "Generic projectable time scaling",
    lambda t, x: jnp.concatenate(
        [jnp.array([time_basis(t)], dtype=jnp.float32), zero_xi()]
    ),
)

add_basis(
    "time_quadratic__t2_dt",
    "Generic projectable time quadratic",
    lambda t, x: jnp.concatenate(
        [jnp.array([time_basis(t) ** 2], dtype=jnp.float32), zero_xi()]
    ),
)


# ============================================================
# Group B: coordinate translations ∂x_i
# ============================================================

for i in range(D):
    add_basis(
        f"translation__dx{i+1}",
        "Generic coordinate translation",
        lambda t, x, i=i: jnp.concatenate(
            [jnp.array([0.0], dtype=jnp.float32), e_i(i)]
        ),
    )


# ============================================================
# Group C: coordinate scalings x_i ∂x_i
# ============================================================

for i in range(D):
    add_basis(
        f"scaling__x{i+1}_dx{i+1}",
        "Generic coordinate scaling",
        lambda t, x, i=i: jnp.concatenate(
            [
                jnp.array([0.0], dtype=jnp.float32),
                jnp.zeros((D,), dtype=jnp.float32).at[i].set(x[i]),
            ]
        ),
    )


# ============================================================
# Group D: global Brownian/Gaeta lifted scaling
# ============================================================

add_basis(
    "global_brownian_scaling__2t_dt_plus_x_dot_dx",
    "50D lift of Brownian/Gaeta scaling",
    lambda t, x: jnp.concatenate(
        [jnp.array([2.0 * time_basis(t)], dtype=jnp.float32), x]
    ),
)


# ============================================================
# Group E: Gaeta-style exponential linear-drift basis
# ============================================================

if CONFIG.get("BASIS_ADD_EXPONENTIAL", False):
    for mu in CONFIG["BASIS_LINEAR_MU_LIST"]:
        mu = float(mu)

        if abs(mu) < 1e-12:
            continue

        # e^{mu t/T} ∂x_i
        for i in range(D):
            add_basis(
                f"exp_translation__mu_{mu:+.2f}__e_mut_dx{i+1}",
                "Gaeta-style exponential translation e^{mu t} ∂x_i",
                lambda t, x, mu=mu, i=i: jnp.concatenate(
                    [
                        jnp.array([0.0], dtype=jnp.float32),
                        jnp.zeros((D,), dtype=jnp.float32).at[i].set(
                            safe_exp(mu * time_basis(t))
                        ),
                    ]
                ),
            )

        # e^{2mu t/T}(∂t + mu x·∇x)
        add_basis(
            f"exp_projective__mu_{mu:+.2f}__e_2mut_dt_plus_mu_x_dot_dx",
            "Gaeta-style projective linear-drift generator",
            lambda t, x, mu=mu: jnp.concatenate(
                [
                    jnp.array(
                        [safe_exp(2.0 * mu * time_basis(t))],
                        dtype=jnp.float32,
                    ),
                    mu * safe_exp(2.0 * mu * time_basis(t)) * x,
                ]
            ),
        )


# ============================================================
# Group F: low-rank rotations and shears
# ============================================================

LOW_RANK_DIRECTIONS = make_low_rank_directions(CONFIG["BASIS_LOW_RANK_K"])
K_LOW = len(LOW_RANK_DIRECTIONS)

print(f"Using {K_LOW} fixed low-rank basis directions.")


# ----------------------------
# Low-rank rotations:
#   (v_b·x)v_a - (v_a·x)v_b
# ----------------------------
if CONFIG.get("BASIS_ADD_LOW_RANK_ROTATIONS", False):
    for a in range(K_LOW):
        for b in range(a + 1, K_LOW):
            va = LOW_RANK_DIRECTIONS[a]
            vb = LOW_RANK_DIRECTIONS[b]

            add_basis(
                f"lowrank_rotation__v{a+1}_v{b+1}",
                "Low-rank antisymmetric rotation generator",
                lambda t, x, va=va, vb=vb: jnp.concatenate(
                    [
                        jnp.array([0.0], dtype=jnp.float32),
                        jnp.dot(vb, x) * va - jnp.dot(va, x) * vb,
                    ]
                ),
            )


# ----------------------------
# Low-rank adjacent shears:
#   (v_b·x)v_a
#
# We use only adjacent directions and both orientations:
#   v_{a+1} -> v_a and v_a -> v_{a+1}
#
# This adds 2(K-1) shears instead of all K(K-1).
# ----------------------------
if CONFIG.get("BASIS_ADD_LOW_RANK_SHEARS", False):
    for a in range(K_LOW - 1):
        va = LOW_RANK_DIRECTIONS[a]
        vb = LOW_RANK_DIRECTIONS[a + 1]

        add_basis(
            f"lowrank_shear__v{a+2}_to_v{a+1}",
            "Low-rank adjacent shear generator",
            lambda t, x, va=va, vb=vb: jnp.concatenate(
                [
                    jnp.array([0.0], dtype=jnp.float32),
                    jnp.dot(vb, x) * va,
                ]
            ),
        )

        add_basis(
            f"lowrank_shear__v{a+1}_to_v{a+2}",
            "Low-rank adjacent shear generator",
            lambda t, x, va=va, vb=vb: jnp.concatenate(
                [
                    jnp.array([0.0], dtype=jnp.float32),
                    jnp.dot(va, x) * vb,
                ]
            ),
        )


# ============================================================
# Finalize basis library
# ============================================================

BASIS_NAMES = [b["name"] for b in BASIS_LIBRARY]
BASIS_SOURCES = [b["source"] for b in BASIS_LIBRARY]
BASIS_FNS = tuple(b["fn"] for b in BASIS_LIBRARY)
N_BASIS = len(BASIS_FNS)

print(f"\nNumber of 50D basis functions: {N_BASIS}")
print(f"BASIS_MODE: {CONFIG['BASIS_MODE']}")

n_minimal = 3 + D + D + 1
n_exp = 0
if CONFIG.get("BASIS_ADD_EXPONENTIAL", False):
    n_exp = len([mu for mu in CONFIG["BASIS_LINEAR_MU_LIST"] if abs(mu) > 1e-12]) * (D + 1)

n_rot = 0
if CONFIG.get("BASIS_ADD_LOW_RANK_ROTATIONS", False):
    n_rot = K_LOW * (K_LOW - 1) // 2

n_shear = 0
if CONFIG.get("BASIS_ADD_LOW_RANK_SHEARS", False):
    n_shear = 2 * (K_LOW - 1)

print("\nBasis count breakdown:")
print(f"  minimal time/translation/scaling/global : {n_minimal}")
print(f"  exponential Gaeta terms                 : {n_exp}")
print(f"  low-rank rotations                      : {n_rot}")
print(f"  low-rank shears                         : {n_shear}")
print(f"  total                                   : {N_BASIS}")

for i, (nm, src) in enumerate(zip(BASIS_NAMES, BASIS_SOURCES)):
    print(f"{i:03d} | {nm} | {src}")


def eval_basis_one(t, x):
    """
    Returns basis values at one point.

    Output shape:
        (R, 51)
    """
    return jnp.stack([fn(t, x) for fn in BASIS_FNS], axis=0)


def eval_generators_point(W, z):
    """
    Evaluate learned generators at one point.

    W shape:
        (m, R)

    z shape:
        (51,) = [t, x1, ..., x50]

    Returns:
        X shape (m, 51), with X[a] = [tau_a, xi_a1, ..., xi_a50].
    """
    t = z[0]
    x = z[1:]
    B = eval_basis_one(t, x)
    return W @ B


def eval_generators_batch(W, tx):
    """
    tx shape:
        (S, 51)

    Returns:
        X shape (S, m, 51)
    """
    return jax.vmap(lambda z: eval_generators_point(W, z))(tx)


# =============================================================================
# Section of Cell 4
# Keep the complete basis-library construction above this point.
#
# Replace the current block beginning at:
#     def _x_box_to_arrays(x_box):
# and ending after:
#     print("\nInitial W shape:", W.shape)
# with the block below.
# =============================================================================

def _x_box_to_arrays(x_box):
    """
    Allow x_box to be either:
        (low_scalar, high_scalar)
    or:
        (low_vector_shape_D, high_vector_shape_D).
    """
    low, high = x_box

    low = jnp.asarray(low, dtype=jnp.float32)
    high = jnp.asarray(high, dtype=jnp.float32)

    if low.ndim == 0:
        low = jnp.full((D,), low, dtype=jnp.float32)
    if high.ndim == 0:
        high = jnp.full((D,), high, dtype=jnp.float32)

    if low.shape != (D,) or high.shape != (D,):
        raise ValueError(
            f"x_box low/high must be scalar or shape ({D},); "
            f"received {low.shape} and {high.shape}."
        )

    if bool(jnp.any(high <= low)):
        raise ValueError("Every x_box upper bound must exceed its lower bound.")

    return low, high


def sample_tx_box(key, n_points, t_box, x_box):
    """
    Sample genuinely ambient points uniformly from
        [t_min,t_max] x x_box^D.

    Returns shape (n_points, D+1).
    """
    n_points = int(n_points)
    if n_points <= 0:
        return jnp.zeros((0, Q), dtype=jnp.float32)

    key_t, key_x = jax.random.split(key)

    t = jax.random.uniform(
        key_t,
        (n_points, 1),
        minval=float(t_box[0]),
        maxval=float(t_box[1]),
    )

    x_low, x_high = _x_box_to_arrays(x_box)
    u = jax.random.uniform(key_x, (n_points, D))
    x = x_low[None, :] + u * (x_high[None, :] - x_low[None, :])

    return jnp.concatenate([t, x], axis=1)


def sample_tx_data(key, dataset, n_points):
    """
    Sample exact observed trajectory states without perturbation.

    Returns shape (n_points, D+1).
    """
    n_points = int(n_points)
    if n_points <= 0:
        return jnp.zeros((0, Q), dtype=jnp.float32)

    t0, x0, _, _ = dataset
    idx = jax.random.randint(key, (n_points,), 0, t0.shape[0])
    return jnp.concatenate([t0[idx, None], x0[idx]], axis=1)


def sample_tx_data_jitter(
    key,
    dataset,
    n_points,
    jitter_std=None,
):
    """
    Sample near-trajectory points using the submitted spatial jitter protocol.

    Time is kept at an observed time. Only x is perturbed.
    """
    n_points = int(n_points)
    if n_points <= 0:
        return jnp.zeros((0, Q), dtype=jnp.float32)

    if jitter_std is None:
        jitter_std = CONFIG["E1_JITTER_STD"]

    key_idx, key_noise = jax.random.split(key)
    t0, x0, _, _ = dataset
    idx = jax.random.randint(key_idx, (n_points,), 0, t0.shape[0])

    t = t0[idx]
    x = x0[idx]
    noise = float(jitter_std) * jax.random.normal(key_noise, x.shape)

    return jnp.concatenate([t[:, None], x + noise], axis=1)


def sample_tx_support_ambient(key, n_points, t_box):
    """
    Optional support-aware ambient sampler.

    Coordinates are sampled independently between empirical training
    quantiles. This remains off the joint trajectory manifold while avoiding
    the extreme corners of the larger configured Cartesian box.
    """
    n_points = int(n_points)
    if n_points <= 0:
        return jnp.zeros((0, Q), dtype=jnp.float32)

    key_t, key_x = jax.random.split(key)
    t = jax.random.uniform(
        key_t,
        (n_points, 1),
        minval=float(t_box[0]),
        maxval=float(t_box[1]),
    )
    u = jax.random.uniform(key_x, (n_points, D))
    x = (
        E1_SUPPORT_X_LOW[None, :]
        + u * (E1_SUPPORT_X_HIGH - E1_SUPPORT_X_LOW)[None, :]
    )
    return jnp.concatenate([t, x], axis=1)


def tx_from_batch_prefix(batch, n_points):
    """
    Use the first n_points from an already-randomized Stage-III data batch.
    """
    n_points = int(n_points)
    if n_points <= 0:
        return jnp.zeros((0, Q), dtype=jnp.float32)

    t_batch, x_batch, _, _ = batch
    if n_points > int(t_batch.shape[0]):
        raise ValueError(
            f"Requested {n_points} trajectory collocation points from a "
            f"batch of size {t_batch.shape[0]}."
        )
    return jnp.concatenate(
        [t_batch[:n_points, None], x_batch[:n_points]],
        axis=1,
    )


def build_e1_symmetry_collocation(branch_name, batch, dataset, key):
    """
    Construct the full fixed-size Stage-III symmetry batch for one E1 branch.

    All returned arrays have shape:
        (CONFIG["E1_TOTAL_SYM_POINTS"], D+1).
    """
    if branch_name not in E1_BRANCH_SPECS:
        raise KeyError(f"Unknown E1 branch: {branch_name}")

    spec = E1_BRANCH_SPECS[branch_name]
    if not spec["use_symmetry"]:
        raise ValueError(
            "NLL-only continuation does not use a symmetry-collocation batch."
        )

    key_jitter, key_box, key_support, key_shuffle = jax.random.split(key, 4)
    pieces = []

    if spec["n_traj"] > 0:
        pieces.append(tx_from_batch_prefix(batch, spec["n_traj"]))

    if spec["n_jitter"] > 0:
        pieces.append(
            sample_tx_data_jitter(
                key_jitter,
                dataset,
                spec["n_jitter"],
                CONFIG["E1_JITTER_STD"],
            )
        )

    if spec["n_box"] > 0:
        pieces.append(
            sample_tx_box(
                key_box,
                spec["n_box"],
                CONFIG["STAGE_F_EXTRA_T_BOX"],
                CONFIG["STAGE_F_EXTRA_X_BOX"],
            )
        )

    if spec["n_support"] > 0:
        pieces.append(
            sample_tx_support_ambient(
                key_support,
                spec["n_support"],
                CONFIG["STAGE_F_EXTRA_T_BOX"],
            )
        )

    if not pieces:
        raise ValueError(f"Symmetry branch {branch_name} has no points.")

    tx = jnp.concatenate(pieces, axis=0)
    expected = int(CONFIG["E1_TOTAL_SYM_POINTS"])

    if int(tx.shape[0]) != expected:
        raise ValueError(
            f"Branch {branch_name} generated {tx.shape[0]} points; "
            f"expected {expected}."
        )

    # The determining loss is a mean, so order should not matter. Shuffling
    # nevertheless prevents any future order-sensitive implementation from
    # treating one source systematically differently.
    perm = jax.random.permutation(key_shuffle, expected)
    return tx[perm]


print("\nValidated E1 Stage-III collocation compositions")
for branch_name, spec in E1_BRANCH_SPECS.items():
    if spec["use_symmetry"]:
        total = (
            spec["n_traj"]
            + spec["n_jitter"]
            + spec["n_box"]
            + spec["n_support"]
        )
        if total != CONFIG["E1_TOTAL_SYM_POINTS"]:
            raise ValueError(
                f"{branch_name} has {total} points, expected "
                f"{CONFIG['E1_TOTAL_SYM_POINTS']}."
            )
        print(
            f"  {branch_name:30s} | total={total:3d} | "
            f"traj={spec['n_traj']:3d} | jitter={spec['n_jitter']:3d} | "
            f"box={spec['n_box']:3d} | support={spec['n_support']:3d}"
        )


# Initialize Stage-II generator coefficients exactly once.
key, subkey = jax.random.split(key)
W = CONFIG["SYM_INIT_SCALE"] * jax.random.normal(
    subkey,
    (CONFIG["N_GENERATORS"], N_BASIS),
)
print("\nInitial W shape:", W.shape)

## Cell 5 — 50D determining loss, global Lie closure, symmetry discovery

In [ ]:
# ============================================================
# Cell 5 — 50D determining loss, global Lie closure, symmetry discovery
# ============================================================

def det_residuals_point(sde_params, W, z):
    """
    50D projectable Ito SDE determining equations.

    Coordinates:
        z = (t, x1, ..., x50)

    Generator:
        X_a = tau_a ∂t + xi_a^i ∂_{x_i}

    Neural SDE:
        dX^i = f^i(t,X) dt + sigma_i(t,X) dW^i

    Diagonal diffusion:
        Sigma^{i alpha} = delta_{i alpha} sigma_i

    Important:
        For the minimal 104-basis, xi is at most linear in x.
        Therefore xi_xx = 0 and the Ito second-derivative term
        in the drift determining equation vanishes.
    """
    f = f_vec_single(sde_params, z)            # (D,)
    sigma = sigma_vec_single(sde_params, z)    # (D,)

    J_f = jax.jacfwd(lambda zz: f_vec_single(sde_params, zz))(z)          # (D, Q)
    J_sigma = jax.jacfwd(lambda zz: sigma_vec_single(sde_params, zz))(z)  # (D, Q)

    f_t = J_f[:, 0]          # (D,)
    f_x = J_f[:, 1:]         # (D, D)

    sigma_t = J_sigma[:, 0]  # (D,)
    sigma_x = J_sigma[:, 1:] # (D, D)

    gen_fun = lambda zz: eval_generators_point(W, zz)

    X = gen_fun(z)                         # (m, Q)
    J = jax.jacfwd(gen_fun)(z)             # (m, Q, Q)

    tau = X[:, 0]                          # (m,)
    xi = X[:, 1:]                          # (m, D)

    tau_t = J[:, 0, 0]                     # (m,)
    tau_x = J[:, 0, 1:]                    # (m, D)

    xi_t = J[:, 1:, 0]                     # (m, D)
    xi_x = J[:, 1:, 1:]                    # (m, D, D)

    adv_xi = jnp.einsum("j,aij->ai", f, xi_x)
    xi_dot_grad_f = jnp.einsum("aj,ij->ai", xi, f_x)
    time_f_tau = tau[:, None] * f_t[None, :] + tau_t[:, None] * f[None, :]

    # Since xi_xx = 0 for the minimal linear basis:
    ito_xi = 0.0

    R_f = (
        xi_t
        + adv_xi
        - xi_dot_grad_f
        - time_f_tau
        + ito_xi
    )  # (m, D)

    # Diffusion residual for diagonal Sigma.
    eye = jnp.eye(D, dtype=jnp.float32)

    first_term = jnp.einsum("p,aip->aip", sigma, xi_x)

    diagonal_correction = (
        jnp.einsum("aj,ij->ai", xi, sigma_x)
        + tau[:, None] * sigma_t[None, :]
        + 0.5 * tau_t[:, None] * sigma[None, :]
    )  # (m, D)

    R_sigma = first_term - diagonal_correction[:, :, None] * eye[None, :, :]

    return R_f, R_sigma, tau_x


def determining_loss(sde_params, W, tx):
    R_f, R_sigma, tau_x = jax.vmap(
        lambda z: det_residuals_point(sde_params, W, z)
    )(tx)

    det = jnp.mean(R_f**2) + jnp.mean(R_sigma**2)
    projectable = jnp.mean(tau_x**2)
    return det, projectable


def generator_jacobians_batch(W, tx):
    """
    Returns:
        X shape (S, m, Q)
        J shape (S, m, Q, Q)
    """
    def one(z):
        gen_fun = lambda zz: eval_generators_point(W, zz)
        X = gen_fun(z)
        J = jax.jacfwd(gen_fun)(z)
        return X, J

    return jax.vmap(one)(tx)


def lie_closure_loss_global(W, tx):
    """
    Global-batch Lie closure in coordinates (t,x1,x2,x3).

        [X_i, X_j] ≈ sum_k C_ij^k X_k

    Structure constants are global because we solve a single least-squares
    problem over the whole collocation batch.
    """
    m = W.shape[0]
    if m < 2:
        return jnp.array(0.0, dtype=jnp.float32)

    X, J = generator_jacobians_batch(W, tx)
    S = tx.shape[0]

    idx_i, idx_j = jnp.triu_indices(m, k=1)
    P = idx_i.shape[0]

    Xi = X[:, idx_i, :]       # (S, P, Q)
    Xj = X[:, idx_j, :]       # (S, P, Q)
    Ji = J[:, idx_i, :, :]    # (S, P, Q, Q)
    Jj = J[:, idx_j, :, :]    # (S, P, Q, Q)

    brackets = (
        jnp.einsum("spc,spqc->spq", Xi, Jj)
        - jnp.einsum("spc,spqc->spq", Xj, Ji)
    )  # (S, P, Q)

    V = jnp.transpose(X, (1, 0, 2)).reshape(m, -1).T      # (Q*S, m)
    B = jnp.transpose(brackets, (1, 0, 2)).reshape(P, -1).T  # (Q*S, P)

    G = V.T @ V
    reg = (CONFIG["CLOSURE_RCOND"] ** 2) * jnp.eye(m, dtype=G.dtype)
    rhs = V.T @ B

    C = jnp.linalg.solve(G + reg, rhs)
    B_proj = V @ C
    residual = B - B_proj

    return jnp.mean(residual**2)


def gram_independence_loss(W, tx):
    """
    Function-space Gram orthonormality:
        G_ab = E[tau_a tau_b + xi_a · xi_b]
    """
    X = eval_generators_batch(W, tx)  # (S, m, Q)
    S = X.shape[0]
    m = X.shape[1]

    V = jnp.transpose(X, (1, 0, 2)).reshape(m, -1).T  # (Q*S, m)
    G = (V.T @ V) / (Q * S)

    I = jnp.eye(m, dtype=G.dtype)
    loss_ind = jnp.mean((G - I) ** 2)

    diag = jnp.diag(G)
    norm_hinge = jnp.mean(
        jnp.maximum(0.0, CONFIG["MIN_GENERATOR_NORM"] - diag) ** 2
    )

    return loss_ind, norm_hinge, G


def sym_discovery_objective(W, sde_params, tx):
    det, proj = determining_loss(sde_params, W, tx)
    close = lie_closure_loss_global(W, tx)
    ind, norm_hinge, _ = gram_independence_loss(W, tx)

    total = (
        CONFIG["LAMBDA_DET"] * det
        + CONFIG["LAMBDA_CLOSE"] * close
        + CONFIG["LAMBDA_IND"] * ind
        + CONFIG["LAMBDA_PROJECTABLE"] * proj
        + CONFIG["LAMBDA_NORM_HINGE"] * norm_hinge
    )

    aux = {
        "total": total,
        "det": det,
        "close": close,
        "ind": ind,
        "projectable": proj,
        "norm_hinge": norm_hinge,
    }
    return total, aux


sym_value_and_grad = jax.value_and_grad(
    lambda W, sde_params, tx: sym_discovery_objective(W, sde_params, tx)[0]
)


@jax.jit
def sym_train_step(W, opt_state, sde_params, tx):
    loss, grads = sym_value_and_grad(W, sde_params, tx)
    W, opt_state = adam_update(
        W,
        grads,
        opt_state,
        CONFIG["SYM_DISCOVERY_LR"],
    )
    _, aux = sym_discovery_objective(W, sde_params, tx)
    return W, opt_state, aux

# =============================================================================
# Section of Cell 5
# Keep Cell 5 from its beginning through the definition of sym_train_step.
# Replace everything from:
#     # Train symmetry basis weights W
# through the end of Cell 5 with the section below.
# =============================================================================

# Freeze the Stage-I checkpoint before symmetry discovery. Stage II may update
# W, but it must never update this checkpoint.
sde_params_baseline = tree_copy(sde_params)

W_opt_state = adam_init(W)
sym_history = []

# Compile once before starting the wall-clock timer.
warm_key = jax.random.fold_in(
    jax.random.PRNGKey(CONFIG["SEED"]),
    51000,
)
tx_warm = sample_tx_data_jitter(
    warm_key,
    train_data,
    CONFIG["SYM_DISCOVERY_POINTS"],
    CONFIG["E1_JITTER_STD"],
)
W_warm, W_opt_warm, aux_warm = sym_train_step(
    W,
    W_opt_state,
    sde_params_baseline,
    tx_warm,
)
tree_block_until_ready((W_warm, W_opt_warm, aux_warm))

sym_discovery_start = time.perf_counter()

for epoch in range(1, CONFIG["SYM_DISCOVERY_EPOCHS"] + 1):
    epoch_key = jax.random.fold_in(
        jax.random.PRNGKey(CONFIG["SEED"]),
        52000 + epoch,
    )
    tx_disc = sample_tx_data_jitter(
        epoch_key,
        train_data,
        CONFIG["SYM_DISCOVERY_POINTS"],
        CONFIG["E1_JITTER_STD"],
    )

    W, W_opt_state, aux = sym_train_step(
        W,
        W_opt_state,
        sde_params_baseline,
        tx_disc,
    )

    aux_np = {name: float(value) for name, value in aux.items()}
    aux_np["epoch"] = epoch
    sym_history.append(aux_np)

    if epoch % CONFIG["PRINT_EVERY"] == 0 or epoch == 1:
        print(
            f"[SymDiscovery] epoch {epoch:5d} | "
            f"total {aux_np['total']: .4e} | "
            f"det {aux_np['det']: .4e} | "
            f"close {aux_np['close']: .4e} | "
            f"ind {aux_np['ind']: .4e} | "
            f"proj {aux_np['projectable']: .4e}"
        )

tree_block_until_ready((W, W_opt_state))
sym_discovery_seconds = time.perf_counter() - sym_discovery_start

# Stop all Stage-III gradients from entering W.
W_frozen = jax.lax.stop_gradient(jnp.array(W))

print(f"Symmetry discovery time (post-compilation): {sym_discovery_seconds:.2f} sec")
print("Learned W shape:", tuple(W_frozen.shape))


# Compact Stage-II diagnostics on three fixed, independently sampled domains.
stage2_diag_root = jax.random.PRNGKey(CONFIG["E1_EVAL_SEED"] + 11)
stage2_diag_keys = jax.random.split(stage2_diag_root, 3)
stage2_diag_sets = {
    "trajectory": sample_tx_data(
        stage2_diag_keys[0],
        train_data,
        CONFIG["SYM_DISCOVERY_POINTS"],
    ),
    "trajectory_jitter": sample_tx_data_jitter(
        stage2_diag_keys[1],
        train_data,
        CONFIG["SYM_DISCOVERY_POINTS"],
        CONFIG["E1_JITTER_STD"],
    ),
    "uniform_box": sample_tx_box(
        stage2_diag_keys[2],
        CONFIG["SYM_DISCOVERY_POINTS"],
        CONFIG["STAGE_F_EXTRA_T_BOX"],
        CONFIG["STAGE_F_EXTRA_X_BOX"],
    ),
}

stage2_diag_rows = []
for domain_name, tx_diag in stage2_diag_sets.items():
    det_value, proj_value = determining_loss(
        sde_params_baseline,
        W_frozen,
        tx_diag,
    )
    _, _, gram_value = gram_independence_loss(W_frozen, tx_diag)
    gram_np = np.asarray(gram_value)
    eigvals = np.clip(np.linalg.eigvalsh(gram_np), 0.0, None)
    eigsum = float(eigvals.sum())
    if eigsum > 0.0:
        probabilities = eigvals / eigsum
        positive = probabilities > 0.0
        effective_rank = float(
            np.exp(-np.sum(probabilities[positive] * np.log(probabilities[positive])))
        )
    else:
        effective_rank = 0.0

    stage2_diag_rows.append(
        {
            "domain": domain_name,
            "det_loss": float(det_value),
            "projectability": float(proj_value),
            "gram_diag_min": float(np.min(np.diag(gram_np))),
            "gram_diag_median": float(np.median(np.diag(gram_np))),
            "gram_diag_max": float(np.max(np.diag(gram_np))),
            "gram_effective_rank": effective_rank,
        }
    )

print("\nStage-II frozen-generator diagnostics")
print(pd.DataFrame(stage2_diag_rows).to_string(index=False))

## Cell 6 — Stage III with controlled symmetry-batch composition

In [ ]:
# =============================================================================
# Cell 6 — Stage III with controlled symmetry-batch composition
# =============================================================================

def stage_f_objective(params, W_fixed, batch, tx_sym):
    """NLL plus determining-equation loss on a supplied fixed-size tx batch."""
    data_loss = data_nll_loss(params, batch)
    det_loss, projectability_loss = determining_loss(params, W_fixed, tx_sym)

    total = (
        data_loss
        + CONFIG["STAGE_F_LAMBDA_SYM_DET"] * det_loss
        + CONFIG["STAGE_F_LAMBDA_PROJECTABLE"] * projectability_loss
    )

    return total, {
        "total": total,
        "data": data_loss,
        "sym_det": det_loss,
        "projectable": projectability_loss,
    }


stage_f_value_and_grad = jax.value_and_grad(
    lambda params, W_fixed, batch, tx_sym: stage_f_objective(
        params,
        W_fixed,
        batch,
        tx_sym,
    )[0]
)


@jax.jit
def stage_f_train_step(params, opt_state, W_fixed, batch, tx_sym):
    _, grads = stage_f_value_and_grad(params, W_fixed, batch, tx_sym)
    params, opt_state = adam_update(
        params,
        grads,
        opt_state,
        CONFIG["STAGE_F_LR"],
    )
    _, aux = stage_f_objective(params, W_fixed, batch, tx_sym)
    return params, opt_state, aux


@jax.jit
def nll_continuation_train_step(params, opt_state, batch):
    loss, grads = jax.value_and_grad(data_nll_loss)(params, batch)
    params, opt_state = adam_update(
        params,
        grads,
        opt_state,
        CONFIG["STAGE_F_LR"],
    )
    post_update_loss = data_nll_loss(params, batch)
    nan_value = jnp.asarray(jnp.nan, dtype=post_update_loss.dtype)
    aux = {
        "total": post_update_loss,
        "data": post_update_loss,
        "sym_det": nan_value,
        "projectable": nan_value,
    }
    return params, opt_state, aux


def e1_epoch_key(stream_id, epoch):
    """Deterministic independent key; stream 61000 is shared data batching."""
    root = jax.random.PRNGKey(CONFIG["SEED"])
    return jax.random.fold_in(jax.random.fold_in(root, int(stream_id)), int(epoch))


def e1_aux_to_host(aux, epoch):
    row = {name: float(value) for name, value in aux.items()}
    row["epoch"] = int(epoch)
    return row


def e1_aux_is_finite(aux_row, use_symmetry):
    required = ["total", "data"]
    if use_symmetry:
        required.extend(["sym_det", "projectable"])
    return all(np.isfinite(aux_row[name]) for name in required)


# Compile both update shapes outside branch timers. Results are discarded.
warm_batch = sample_batch(
    e1_epoch_key(61000, 0),
    train_data,
    CONFIG["STAGE_F_BATCH_SIZE"],
)
warm_sym_tx = build_e1_symmetry_collocation(
    "sym_traj_plus_jitter",
    warm_batch,
    train_data,
    e1_epoch_key(62000, 0),
)
warm_params = tree_copy(sde_params_baseline)
warm_opt = adam_init(warm_params)
warm_out = stage_f_train_step(
    warm_params,
    warm_opt,
    W_frozen,
    warm_batch,
    warm_sym_tx,
)
tree_block_until_ready(warm_out)
warm_out = nll_continuation_train_step(warm_params, warm_opt, warm_batch)
tree_block_until_ready(warm_out)


# Every Stage-III branch starts from exactly the same frozen Stage-I checkpoint.
e1_models = {"stage1_baseline": tree_copy(sde_params_baseline)}
e1_histories = {"stage1_baseline": []}
e1_train_seconds = {"stage1_baseline": 0.0}
e1_status = {"stage1_baseline": "reference"}
e1_completed_epochs = {"stage1_baseline": 0}
e1_failure_epoch = {"stage1_baseline": None}


for branch_index, (branch_name, spec) in enumerate(E1_BRANCH_SPECS.items()):
    print("\n" + "=" * 78)
    print(f"E1 Stage-III branch: {branch_name} — {spec['label']}")
    print("=" * 78)

    branch_params = tree_copy(sde_params_baseline)
    branch_opt_state = adam_init(branch_params)
    branch_history = []
    branch_status = "completed"
    failure_epoch = None

    tree_block_until_ready((branch_params, branch_opt_state))
    branch_start = time.perf_counter()

    for epoch in range(1, CONFIG["STAGE_F_EPOCHS"] + 1):
        # The data-batch stream is identical for every branch.
        batch = sample_batch(
            e1_epoch_key(61000, epoch),
            train_data,
            CONFIG["STAGE_F_BATCH_SIZE"],
        )

        previous_params = branch_params
        previous_opt_state = branch_opt_state

        if spec["use_symmetry"]:
            tx_sym = build_e1_symmetry_collocation(
                branch_name,
                batch,
                train_data,
                e1_epoch_key(63000 + branch_index, epoch),
            )
            candidate_params, candidate_opt_state, aux = stage_f_train_step(
                branch_params,
                branch_opt_state,
                W_frozen,
                batch,
                tx_sym,
            )
        else:
            candidate_params, candidate_opt_state, aux = (
                nll_continuation_train_step(
                    branch_params,
                    branch_opt_state,
                    batch,
                )
            )

        # Converting aux to Python synchronizes the update and enables the
        # non-finite guard. If an update fails, retain the last finite state.
        aux_row = e1_aux_to_host(aux, epoch)
        if not e1_aux_is_finite(aux_row, spec["use_symmetry"]):
            branch_params = previous_params
            branch_opt_state = previous_opt_state
            branch_status = "stopped_nonfinite"
            failure_epoch = epoch
            print(
                f"[{branch_name}] stopped at epoch {epoch}: "
                "non-finite objective; last finite parameters retained."
            )
            break

        branch_params = candidate_params
        branch_opt_state = candidate_opt_state
        branch_history.append(aux_row)

        if epoch % CONFIG["PRINT_EVERY"] == 0 or epoch == 1:
            if spec["use_symmetry"]:
                print(
                    f"[{branch_name}] epoch {epoch:4d} | "
                    f"total {aux_row['total']: .5e} | "
                    f"nll {aux_row['data']: .5e} | "
                    f"det {aux_row['sym_det']: .5e}"
                )
            else:
                print(
                    f"[{branch_name}] epoch {epoch:4d} | "
                    f"nll {aux_row['data']: .5e}"
                )

    tree_block_until_ready((branch_params, branch_opt_state))
    branch_seconds = time.perf_counter() - branch_start

    e1_models[branch_name] = tree_copy(branch_params)
    e1_histories[branch_name] = branch_history
    e1_train_seconds[branch_name] = float(branch_seconds)
    e1_status[branch_name] = branch_status
    e1_completed_epochs[branch_name] = len(branch_history)
    e1_failure_epoch[branch_name] = failure_epoch

    print(
        f"[{branch_name}] status={branch_status} | "
        f"completed_epochs={len(branch_history)} | "
        f"time={branch_seconds:.2f} sec"
    )


E1_MODEL_ORDER = ["stage1_baseline"] + list(E1_BRANCH_SPECS.keys())

print("\nE1 branch training summary")
print(
    pd.DataFrame(
        [
            {
                "model": model_name,
                "label": E1_MODEL_LABELS[model_name],
                "status": e1_status[model_name],
                "completed_epochs": e1_completed_epochs[model_name],
                "failure_epoch": e1_failure_epoch[model_name],
                "stage3_seconds": e1_train_seconds[model_name],
            }
            for model_name in E1_MODEL_ORDER
        ]
    ).to_string(index=False)
)


## Cell 7 — Numerical evaluation and reporting

In [ ]:
# =============================================================================
# Cell 7 — Numerical evaluation and reporting
#
# This cell deliberately produces numerical results only. It prints:
#   1. a run manifest;
#   2. collocation-geometry diagnostics;
#   3. drift/diffusion coefficient errors on fixed evaluation domains;
#   4. train/test NLL and increment-calibration diagnostics;
#   5. pointwise determining-residual distributions;
#   6. runtime/stability information;
#   7. direct ambient-versus-jitter comparisons and directional flags;
#   8. CSV blocks and a JSON bundle for aggregation across SDEs/seeds.
# =============================================================================

def e1_safe_percent_reduction(new_value, reference_value, eps=1e-30):
    """Positive means new_value is lower (better) than reference_value."""
    if not (np.isfinite(new_value) and np.isfinite(reference_value)):
        return np.nan
    if abs(reference_value) <= eps:
        return np.nan
    return 100.0 * (reference_value - new_value) / abs(reference_value)


def e1_distribution_summary(values):
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return {
            "n": 0,
            "mean": np.nan,
            "std": np.nan,
            "se": np.nan,
            "min": np.nan,
            "p10": np.nan,
            "p25": np.nan,
            "median": np.nan,
            "p75": np.nan,
            "p90": np.nan,
            "p95": np.nan,
            "p99": np.nan,
            "max": np.nan,
        }

    std = float(np.std(finite, ddof=1)) if finite.size > 1 else 0.0
    quantiles = np.quantile(
        finite,
        [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99],
    )
    return {
        "n": int(finite.size),
        "mean": float(np.mean(finite)),
        "std": std,
        "se": float(std / np.sqrt(finite.size)),
        "min": float(np.min(finite)),
        "p10": float(quantiles[0]),
        "p25": float(quantiles[1]),
        "median": float(quantiles[2]),
        "p75": float(quantiles[3]),
        "p90": float(quantiles[4]),
        "p95": float(quantiles[5]),
        "p99": float(quantiles[6]),
        "max": float(np.max(finite)),
    }


def e1_error_metrics(prediction, target, prefix):
    prediction = np.asarray(prediction, dtype=np.float64)
    target = np.asarray(target, dtype=np.float64)
    error = prediction - target

    mse = float(np.mean(error**2))
    rmse = float(np.sqrt(mse))
    mae = float(np.mean(np.abs(error)))

    target_norm = float(np.linalg.norm(target.reshape(-1)))
    if target_norm > 1e-30:
        relative_l2 = float(
            np.linalg.norm(error.reshape(-1)) / target_norm
        )
    else:
        relative_l2 = np.nan

    return {
        f"{prefix}_mse": mse,
        f"{prefix}_rmse": rmse,
        f"{prefix}_mae": mae,
        f"{prefix}_relative_l2": relative_l2,
    }


@jax.jit
def e1_surrogate_values(params, tx):
    return jax.vmap(
        lambda z: surrogate_f_sigma(params, z[0], z[1:])
    )(tx)


@jax.jit
def e1_true_values(tx):
    true_f = jax.vmap(lambda z: true_drift(z[0], z[1:]))(tx)
    true_s = jax.vmap(lambda z: true_sigma(z[0], z[1:]))(tx)
    return true_f, true_s


def e1_coefficient_metrics(params, tx):
    pred_f, pred_s = e1_surrogate_values(params, tx)
    true_f, true_s = e1_true_values(tx)
    tree_block_until_ready((pred_f, pred_s, true_f, true_s))

    metrics = {}
    metrics.update(
        e1_error_metrics(
            np.asarray(pred_f),
            np.asarray(true_f),
            "drift",
        )
    )
    metrics.update(
        e1_error_metrics(
            np.asarray(pred_s),
            np.asarray(true_s),
            "sigma",
        )
    )
    return metrics


@jax.jit
def e1_increment_arrays(params, dataset):
    t0, x0, dt, dx = dataset
    pred_f, pred_s = jax.vmap(
        lambda tt, xx: surrogate_f_sigma(params, tt, xx)
    )(t0, x0)

    mean_dx = pred_f * dt[:, None]
    variance_dx = pred_s**2 * dt[:, None] + 1e-8
    standardized_innovation = (dx - mean_dx) / jnp.sqrt(variance_dx)
    return mean_dx, standardized_innovation


def e1_dataset_diagnostics(params, dataset):
    nll = float(data_nll_loss(params, dataset))
    mean_dx, standardized_innovation = e1_increment_arrays(params, dataset)
    tree_block_until_ready((mean_dx, standardized_innovation))

    observed_dx = np.asarray(dataset[3], dtype=np.float64)
    predicted_mean_dx = np.asarray(mean_dx, dtype=np.float64)
    innovation = np.asarray(standardized_innovation, dtype=np.float64)
    increment_error = predicted_mean_dx - observed_dx

    return {
        "nll": nll,
        "nll_per_dimension": float(nll / D),
        "increment_mean_mse": float(np.mean(increment_error**2)),
        "increment_mean_rmse": float(
            np.sqrt(np.mean(increment_error**2))
        ),
        "increment_mean_mae": float(np.mean(np.abs(increment_error))),
        # For a calibrated diagonal Gaussian transition model, the following
        # two quantities should approach 0 and 1, respectively.
        "standardized_innovation_mean": float(np.mean(innovation)),
        "standardized_innovation_second_moment": float(
            np.mean(innovation**2)
        ),
    }


def e1_standardized_nearest_neighbor_distances(query_x):
    """
    Euclidean nearest-neighbour distance after coordinatewise standardization
    by training-state standard deviations.
    """
    query_x = jnp.asarray(query_x, dtype=jnp.float32)
    reference_x = E1_DISTANCE_REFERENCE_X
    scale = E1_TRAIN_X_STD

    pieces = []
    chunk_size = CONFIG["E1_DISTANCE_CHUNK_SIZE"]

    for start in range(0, int(query_x.shape[0]), chunk_size):
        stop = min(start + chunk_size, int(query_x.shape[0]))
        query_chunk = query_x[start:stop]
        differences = (
            query_chunk[:, None, :] - reference_x[None, :, :]
        ) / scale[None, None, :]
        squared_distances = jnp.sum(differences**2, axis=-1)
        nearest = jnp.sqrt(jnp.min(squared_distances, axis=1))
        pieces.append(np.asarray(nearest))

    return np.concatenate(pieces, axis=0)


@jax.jit
def e1_pointwise_residual_chunk(params, W_fixed, tx_chunk):
    residual_f, residual_sigma, tau_x = jax.vmap(
        lambda z: det_residuals_point(params, W_fixed, z)
    )(tx_chunk)

    det_per_point = (
        jnp.mean(residual_f**2, axis=(1, 2))
        + jnp.mean(residual_sigma**2, axis=(1, 2, 3))
    )
    projectability_per_point = jnp.mean(tau_x**2, axis=(1, 2))
    return det_per_point, projectability_per_point


def e1_pointwise_residuals(params, tx):
    """
    Evaluate fixed-shape chunks to control memory. A final short chunk is padded
    only for compilation-shape consistency, then sliced back to valid points.
    """
    tx = jnp.asarray(tx, dtype=jnp.float32)
    n_points = int(tx.shape[0])
    chunk_size = int(CONFIG["E1_RESIDUAL_CHUNK_SIZE"])

    det_pieces = []
    projectability_pieces = []

    for start in range(0, n_points, chunk_size):
        stop = min(start + chunk_size, n_points)
        tx_chunk = tx[start:stop]
        n_valid = stop - start

        if n_valid < chunk_size:
            padding = jnp.repeat(
                tx_chunk[-1:, :],
                chunk_size - n_valid,
                axis=0,
            )
            tx_chunk = jnp.concatenate([tx_chunk, padding], axis=0)

        det_values, projectability_values = e1_pointwise_residual_chunk(
            params,
            W_frozen,
            tx_chunk,
        )
        tree_block_until_ready((det_values, projectability_values))
        det_pieces.append(np.asarray(det_values[:n_valid]))
        projectability_pieces.append(
            np.asarray(projectability_values[:n_valid])
        )

    return (
        np.concatenate(det_pieces, axis=0),
        np.concatenate(projectability_pieces, axis=0),
    )


def e1_add_reference_reductions(rows, metric_names):
    """Add within-domain reductions versus Stage I and NLL continuation."""
    lookup = {
        (row["model"], row["domain"]): row
        for row in rows
    }

    for row in rows:
        domain = row["domain"]
        stage1_row = lookup[("stage1_baseline", domain)]
        nll_row = lookup[("nll_continuation", domain)]

        for metric_name in metric_names:
            row[f"{metric_name}_reduction_vs_stage1_pct"] = (
                e1_safe_percent_reduction(
                    row[metric_name],
                    stage1_row[metric_name],
                )
            )
            row[f"{metric_name}_reduction_vs_nll_continuation_pct"] = (
                e1_safe_percent_reduction(
                    row[metric_name],
                    nll_row[metric_name],
                )
            )


def e1_json_ready(value):
    if isinstance(value, dict):
        return {str(k): e1_json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [e1_json_ready(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, np.ndarray):
        return e1_json_ready(value.tolist())
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


# -----------------------------------------------------------------------------
# Fixed, training-independent evaluation domains
# -----------------------------------------------------------------------------
eval_root = jax.random.PRNGKey(CONFIG["E1_EVAL_SEED"])
eval_keys = jax.random.split(eval_root, 16)

coefficient_sets = {
    "trajectory_test": sample_tx_data(
        eval_keys[0],
        test_data,
        CONFIG["E1_COEFF_EVAL_POINTS"],
    ),
    "trajectory_test_jitter": sample_tx_data_jitter(
        eval_keys[1],
        test_data,
        CONFIG["E1_COEFF_EVAL_POINTS"],
        CONFIG["E1_JITTER_STD"],
    ),
    "uniform_box": sample_tx_box(
        eval_keys[2],
        CONFIG["E1_COEFF_EVAL_POINTS"],
        CONFIG["STAGE_F_EXTRA_T_BOX"],
        CONFIG["STAGE_F_EXTRA_X_BOX"],
    ),
}

residual_sets = {
    "trajectory_test": sample_tx_data(
        eval_keys[3],
        test_data,
        CONFIG["E1_RESIDUAL_EVAL_POINTS"],
    ),
    "trajectory_test_jitter": sample_tx_data_jitter(
        eval_keys[4],
        test_data,
        CONFIG["E1_RESIDUAL_EVAL_POINTS"],
        CONFIG["E1_JITTER_STD"],
    ),
    "uniform_box": sample_tx_box(
        eval_keys[5],
        CONFIG["E1_RESIDUAL_EVAL_POINTS"],
        CONFIG["STAGE_F_EXTRA_T_BOX"],
        CONFIG["STAGE_F_EXTRA_X_BOX"],
    ),
}

geometry_sets = {
    "trajectory_ondata_disjoint": E1_DISTANCE_ONDATA_PROBE_X,
    "trajectory_jitter": sample_tx_data_jitter(
        eval_keys[6],
        train_data,
        CONFIG["E1_DISTANCE_PROBE_POINTS"],
        CONFIG["E1_JITTER_STD"],
    )[:, 1:],
    "uniform_box": sample_tx_box(
        eval_keys[7],
        CONFIG["E1_DISTANCE_PROBE_POINTS"],
        CONFIG["STAGE_F_EXTRA_T_BOX"],
        CONFIG["STAGE_F_EXTRA_X_BOX"],
    )[:, 1:],
}

if CONFIG["E1_ENABLE_SUPPORT_AMBIENT"]:
    coefficient_sets["support_ambient"] = sample_tx_support_ambient(
        eval_keys[8],
        CONFIG["E1_COEFF_EVAL_POINTS"],
        CONFIG["STAGE_F_EXTRA_T_BOX"],
    )
    residual_sets["support_ambient"] = sample_tx_support_ambient(
        eval_keys[9],
        CONFIG["E1_RESIDUAL_EVAL_POINTS"],
        CONFIG["STAGE_F_EXTRA_T_BOX"],
    )
    geometry_sets["support_ambient"] = sample_tx_support_ambient(
        eval_keys[10],
        CONFIG["E1_DISTANCE_PROBE_POINTS"],
        CONFIG["STAGE_F_EXTRA_T_BOX"],
    )[:, 1:]


# -----------------------------------------------------------------------------
# 1. Collocation geometry
# -----------------------------------------------------------------------------
geometry_distances = {
    domain_name: e1_standardized_nearest_neighbor_distances(x_values)
    for domain_name, x_values in geometry_sets.items()
}
ondata_p95 = float(
    np.quantile(
        geometry_distances["trajectory_ondata_disjoint"],
        0.95,
    )
)

geometry_rows = []
for domain_name, distances in geometry_distances.items():
    row = {
        "sde": CONFIG["TRUE_SDE_NAME"],
        "seed": CONFIG["SEED"],
        "domain": domain_name,
        **e1_distribution_summary(distances),
    }
    row["ondata_p95_threshold"] = ondata_p95
    row["fraction_beyond_ondata_p95"] = float(
        np.mean(np.asarray(distances) > ondata_p95)
    )
    geometry_rows.append(row)


# -----------------------------------------------------------------------------
# 2. Coefficient recovery on the same fixed domains for every model
# -----------------------------------------------------------------------------
coefficient_rows = []
for model_name in E1_MODEL_ORDER:
    for domain_name, tx_eval in coefficient_sets.items():
        coefficient_rows.append(
            {
                "sde": CONFIG["TRUE_SDE_NAME"],
                "seed": CONFIG["SEED"],
                "model": model_name,
                "label": E1_MODEL_LABELS[model_name],
                "domain": domain_name,
                **e1_coefficient_metrics(e1_models[model_name], tx_eval),
            }
        )

e1_add_reference_reductions(
    coefficient_rows,
    [
        "drift_mse",
        "drift_rmse",
        "drift_mae",
        "drift_relative_l2",
        "sigma_mse",
        "sigma_rmse",
        "sigma_mae",
        "sigma_relative_l2",
    ],
)


# -----------------------------------------------------------------------------
# 3. Data fit and one-step transition calibration
# -----------------------------------------------------------------------------
nll_rows = []
for model_name in E1_MODEL_ORDER:
    train_metrics = e1_dataset_diagnostics(
        e1_models[model_name],
        train_data,
    )
    test_metrics = e1_dataset_diagnostics(
        e1_models[model_name],
        test_data,
    )

    row = {
        "sde": CONFIG["TRUE_SDE_NAME"],
        "seed": CONFIG["SEED"],
        "model": model_name,
        "label": E1_MODEL_LABELS[model_name],
        **{f"train_{name}": value for name, value in train_metrics.items()},
        **{f"test_{name}": value for name, value in test_metrics.items()},
    }
    nll_rows.append(row)

nll_lookup = {row["model"]: row for row in nll_rows}
for row in nll_rows:
    for metric_name in [
        "train_nll",
        "test_nll",
        "train_increment_mean_mse",
        "test_increment_mean_mse",
    ]:
        row[f"{metric_name}_reduction_vs_stage1_pct"] = (
            e1_safe_percent_reduction(
                row[metric_name],
                nll_lookup["stage1_baseline"][metric_name],
            )
        )
        row[f"{metric_name}_reduction_vs_nll_continuation_pct"] = (
            e1_safe_percent_reduction(
                row[metric_name],
                nll_lookup["nll_continuation"][metric_name],
            )
        )


# -----------------------------------------------------------------------------
# 4. Pointwise determining-equation residual distributions
# -----------------------------------------------------------------------------
residual_rows = []
for model_name in E1_MODEL_ORDER:
    print(f"Evaluating pointwise residuals: {model_name}")
    for domain_name, tx_eval in residual_sets.items():
        det_values, projectability_values = e1_pointwise_residuals(
            e1_models[model_name],
            tx_eval,
        )
        det_summary = e1_distribution_summary(det_values)
        proj_summary = e1_distribution_summary(projectability_values)

        residual_rows.append(
            {
                "sde": CONFIG["TRUE_SDE_NAME"],
                "seed": CONFIG["SEED"],
                "model": model_name,
                "label": E1_MODEL_LABELS[model_name],
                "domain": domain_name,
                **{
                    f"det_{name}": value
                    for name, value in det_summary.items()
                },
                **{
                    f"projectability_{name}": value
                    for name, value in proj_summary.items()
                },
            }
        )

e1_add_reference_reductions(
    residual_rows,
    [
        "det_mean",
        "det_median",
        "det_p90",
        "det_p95",
    ],
)


# -----------------------------------------------------------------------------
# 5. Runtime and stability
# -----------------------------------------------------------------------------
runtime_rows = []
for model_name in E1_MODEL_ORDER:
    history = e1_histories[model_name]
    finite_sym_values = [
        row["sym_det"]
        for row in history
        if np.isfinite(row.get("sym_det", np.nan))
    ]
    final_row = history[-1] if history else {}

    runtime_rows.append(
        {
            "sde": CONFIG["TRUE_SDE_NAME"],
            "seed": CONFIG["SEED"],
            "model": model_name,
            "label": E1_MODEL_LABELS[model_name],
            "status": e1_status[model_name],
            "completed_epochs": e1_completed_epochs[model_name],
            "failure_epoch": e1_failure_epoch[model_name],
            "stage3_seconds": e1_train_seconds[model_name],
            "seconds_per_completed_epoch": (
                e1_train_seconds[model_name]
                / e1_completed_epochs[model_name]
                if e1_completed_epochs[model_name] > 0
                else np.nan
            ),
            "final_stage3_total": final_row.get("total", np.nan),
            "final_stage3_data_nll": final_row.get("data", np.nan),
            "final_stage3_sym_det": final_row.get("sym_det", np.nan),
            "min_stage3_sym_det": (
                float(np.min(finite_sym_values))
                if finite_sym_values
                else np.nan
            ),
            "max_stage3_sym_det": (
                float(np.max(finite_sym_values))
                if finite_sym_values
                else np.nan
            ),
        }
    )


# -----------------------------------------------------------------------------
# 6. Compact primary table and direct ambient-versus-controls comparisons
# -----------------------------------------------------------------------------
coefficient_lookup = {
    (row["model"], row["domain"]): row
    for row in coefficient_rows
}
residual_lookup = {
    (row["model"], row["domain"]): row
    for row in residual_rows
}
runtime_lookup = {row["model"]: row for row in runtime_rows}

primary_rows = []
for model_name in E1_MODEL_ORDER:
    coeff = coefficient_lookup[(model_name, "uniform_box")]
    resid = residual_lookup[(model_name, "uniform_box")]
    fit = nll_lookup[model_name]
    run = runtime_lookup[model_name]

    primary_rows.append(
        {
            "sde": CONFIG["TRUE_SDE_NAME"],
            "seed": CONFIG["SEED"],
            "model": model_name,
            "drift_box_mse": coeff["drift_mse"],
            "sigma_box_mse": coeff["sigma_mse"],
            "box_det_mean": resid["det_mean"],
            "box_det_p95": resid["det_p95"],
            "test_nll": fit["test_nll"],
            "test_nll_per_dimension": fit["test_nll_per_dimension"],
            "test_innovation_second_moment": (
                fit["test_standardized_innovation_second_moment"]
            ),
            "drift_box_reduction_vs_stage1_pct": (
                coeff["drift_mse_reduction_vs_stage1_pct"]
            ),
            "drift_box_reduction_vs_nll_pct": (
                coeff["drift_mse_reduction_vs_nll_continuation_pct"]
            ),
            "sigma_box_reduction_vs_stage1_pct": (
                coeff["sigma_mse_reduction_vs_stage1_pct"]
            ),
            "box_det_reduction_vs_stage1_pct": (
                resid["det_mean_reduction_vs_stage1_pct"]
            ),
            "box_det_reduction_vs_nll_pct": (
                resid["det_mean_reduction_vs_nll_continuation_pct"]
            ),
            "status": run["status"],
            "stage3_seconds": run["stage3_seconds"],
        }
    )


ambient_models = [
    model_name
    for model_name in [
        "sym_traj_plus_box",
        "sym_box_only",
        "sym_traj_jitter_box",
        "sym_traj_plus_support_ambient",
    ]
    if model_name in e1_models
]

pairwise_rows = []
for model_name in ambient_models:
    ambient_coeff = coefficient_lookup[(model_name, "uniform_box")]
    jitter_coeff = coefficient_lookup[
        ("sym_traj_plus_jitter", "uniform_box")
    ]
    nll_coeff = coefficient_lookup[("nll_continuation", "uniform_box")]

    ambient_resid = residual_lookup[(model_name, "uniform_box")]
    jitter_resid = residual_lookup[
        ("sym_traj_plus_jitter", "uniform_box")
    ]
    nll_resid = residual_lookup[("nll_continuation", "uniform_box")]

    ambient_nll = nll_lookup[model_name]["test_nll"]
    jitter_nll = nll_lookup["sym_traj_plus_jitter"]["test_nll"]
    continuation_nll = nll_lookup["nll_continuation"]["test_nll"]

    pairwise_rows.append(
        {
            "sde": CONFIG["TRUE_SDE_NAME"],
            "seed": CONFIG["SEED"],
            "ambient_model": model_name,
            "drift_box_improvement_vs_jitter_pct": (
                e1_safe_percent_reduction(
                    ambient_coeff["drift_mse"],
                    jitter_coeff["drift_mse"],
                )
            ),
            "drift_box_improvement_vs_nll_pct": (
                e1_safe_percent_reduction(
                    ambient_coeff["drift_mse"],
                    nll_coeff["drift_mse"],
                )
            ),
            "sigma_box_improvement_vs_jitter_pct": (
                e1_safe_percent_reduction(
                    ambient_coeff["sigma_mse"],
                    jitter_coeff["sigma_mse"],
                )
            ),
            "sigma_box_improvement_vs_nll_pct": (
                e1_safe_percent_reduction(
                    ambient_coeff["sigma_mse"],
                    nll_coeff["sigma_mse"],
                )
            ),
            "box_det_improvement_vs_jitter_pct": (
                e1_safe_percent_reduction(
                    ambient_resid["det_mean"],
                    jitter_resid["det_mean"],
                )
            ),
            "box_det_improvement_vs_nll_pct": (
                e1_safe_percent_reduction(
                    ambient_resid["det_mean"],
                    nll_resid["det_mean"],
                )
            ),
            # Negative deltas mean the ambient model has lower test NLL.
            "test_nll_delta_vs_jitter": ambient_nll - jitter_nll,
            "test_nll_delta_vs_nll_continuation": (
                ambient_nll - continuation_nll
            ),
        }
    )


# -----------------------------------------------------------------------------
# 7. Single-run directional flags
# These are transparent checks, not inferential statistics. Aggregate the CSV
# output over at least 3 (ideally 5) seeds before drawing conclusions.
# -----------------------------------------------------------------------------
geometry_lookup = {row["domain"]: row for row in geometry_rows}
directional_flags = [
    {
        "sde": CONFIG["TRUE_SDE_NAME"],
        "seed": CONFIG["SEED"],
        "flag": "uniform_box_is_off_manifold",
        "value": (
            geometry_lookup["uniform_box"]["fraction_beyond_ondata_p95"]
            >= CONFIG["E1_OFF_MANIFOLD_FRACTION_THRESHOLD"]
        ),
        "observed": geometry_lookup["uniform_box"][
            "fraction_beyond_ondata_p95"
        ],
        "threshold_or_tolerance": (
            CONFIG["E1_OFF_MANIFOLD_FRACTION_THRESHOLD"]
        ),
    }
]

for model_name in ambient_models:
    coeff = coefficient_lookup[(model_name, "uniform_box")]
    jitter_coeff = coefficient_lookup[
        ("sym_traj_plus_jitter", "uniform_box")
    ]
    resid = residual_lookup[(model_name, "uniform_box")]
    jitter_resid = residual_lookup[
        ("sym_traj_plus_jitter", "uniform_box")
    ]

    ambient_test_nll = nll_lookup[model_name]["test_nll"]
    jitter_test_nll = nll_lookup["sym_traj_plus_jitter"]["test_nll"]
    allowed_nll_increase = (
        CONFIG["E1_NLL_REL_TOL"] * max(abs(jitter_test_nll), 1e-12)
    )

    checks = {
        f"{model_name}:lower_box_drift_mse_than_jitter": (
            coeff["drift_mse"] < jitter_coeff["drift_mse"],
            coeff["drift_mse"] - jitter_coeff["drift_mse"],
            0.0,
        ),
        f"{model_name}:lower_box_sigma_mse_than_jitter": (
            coeff["sigma_mse"] < jitter_coeff["sigma_mse"],
            coeff["sigma_mse"] - jitter_coeff["sigma_mse"],
            0.0,
        ),
        f"{model_name}:lower_box_det_mean_than_jitter": (
            resid["det_mean"] < jitter_resid["det_mean"],
            resid["det_mean"] - jitter_resid["det_mean"],
            0.0,
        ),
        f"{model_name}:test_nll_within_5pct_of_jitter": (
            ambient_test_nll <= jitter_test_nll + allowed_nll_increase,
            ambient_test_nll - jitter_test_nll,
            allowed_nll_increase,
        ),
    }

    for flag_name, (flag_value, observed, tolerance) in checks.items():
        directional_flags.append(
            {
                "sde": CONFIG["TRUE_SDE_NAME"],
                "seed": CONFIG["SEED"],
                "flag": flag_name,
                "value": bool(flag_value),
                "observed": float(observed),
                "threshold_or_tolerance": float(tolerance),
            }
        )


# -----------------------------------------------------------------------------
# 8. Human-readable output
# -----------------------------------------------------------------------------
geometry_df = pd.DataFrame(geometry_rows)
coefficient_df = pd.DataFrame(coefficient_rows)
nll_df = pd.DataFrame(nll_rows)
residual_df = pd.DataFrame(residual_rows)
runtime_df = pd.DataFrame(runtime_rows)
primary_df = pd.DataFrame(primary_rows)
pairwise_df = pd.DataFrame(pairwise_rows)
directional_df = pd.DataFrame(directional_flags)

run_manifest = {
    "experiment": "E1_stage3_collocation_distribution",
    "sde": CONFIG["TRUE_SDE_NAME"],
    "manuscript_example": E1_EXAMPLE_LABEL,
    "seed": CONFIG["SEED"],
    "state_dimension": D,
    "n_trajectories": CONFIG["N_TRAJECTORIES"],
    "train_increment_count": int(train_data[0].shape[0]),
    "test_increment_count": int(test_data[0].shape[0]),
    "stage1_epochs": CONFIG["BASELINE_EPOCHS"],
    "stage2_epochs": CONFIG["SYM_DISCOVERY_EPOCHS"],
    "stage2_jitter_std": CONFIG["E1_JITTER_STD"],
    "stage2_seconds_post_compilation": sym_discovery_seconds,
    "stage3_epochs": CONFIG["STAGE_F_EPOCHS"],
    "stage3_lr": CONFIG["STAGE_F_LR"],
    "stage3_lambda_sym_det": CONFIG["STAGE_F_LAMBDA_SYM_DET"],
    "stage3_lambda_projectable": CONFIG[
        "STAGE_F_LAMBDA_PROJECTABLE"
    ],
    "symmetry_points_per_update": CONFIG["E1_TOTAL_SYM_POINTS"],
    "coefficient_eval_points_per_domain": CONFIG[
        "E1_COEFF_EVAL_POINTS"
    ],
    "residual_eval_points_per_domain": CONFIG[
        "E1_RESIDUAL_EVAL_POINTS"
    ],
    "branch_specs": E1_BRANCH_SPECS,
    "branch_status": e1_status,
}

print("\n" + "=" * 100)
print("E1 RUN MANIFEST")
print("=" * 100)
print(json.dumps(e1_json_ready(run_manifest), indent=2, sort_keys=True))

print("\n" + "=" * 100)
print("COLLOCATION GEOMETRY: STANDARDIZED NEAREST-NEIGHBOUR DISTANCE")
print("=" * 100)
print(
    geometry_df[
        [
            "domain",
            "n",
            "mean",
            "std",
            "median",
            "p90",
            "p95",
            "p99",
            "fraction_beyond_ondata_p95",
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 100)
print("PRIMARY E1 TABLE: UNIFORM-BOX GENERALIZATION, TEST NLL, AND STABILITY")
print("=" * 100)
print(primary_df.to_string(index=False))

print("\n" + "=" * 100)
print("COEFFICIENT METRICS: ALL FIXED EVALUATION DOMAINS")
print("=" * 100)
print(
    coefficient_df[
        [
            "model",
            "domain",
            "drift_mse",
            "drift_mae",
            "drift_relative_l2",
            "sigma_mse",
            "sigma_mae",
            "sigma_relative_l2",
            "drift_mse_reduction_vs_stage1_pct",
            "drift_mse_reduction_vs_nll_continuation_pct",
            "sigma_mse_reduction_vs_stage1_pct",
            "sigma_mse_reduction_vs_nll_continuation_pct",
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 100)
print("TRAIN/TEST NLL AND ONE-STEP CALIBRATION")
print("=" * 100)
print(nll_df.to_string(index=False))

print("\n" + "=" * 100)
print("POINTWISE DETERMINING-RESIDUAL DISTRIBUTIONS")
print("=" * 100)
print(
    residual_df[
        [
            "model",
            "domain",
            "det_mean",
            "det_se",
            "det_median",
            "det_p90",
            "det_p95",
            "det_p99",
            "det_mean_reduction_vs_stage1_pct",
            "det_mean_reduction_vs_nll_continuation_pct",
            "projectability_mean",
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 100)
print("RUNTIME / NUMERICAL STABILITY")
print("=" * 100)
print(runtime_df.to_string(index=False))

print("\n" + "=" * 100)
print("DIRECT AMBIENT-COLLOCATION COMPARISONS")
print("=" * 100)
if len(pairwise_df) > 0:
    print(pairwise_df.to_string(index=False))
else:
    print("No ambient branch was enabled.")

print("\n" + "=" * 100)
print("SINGLE-RUN DIRECTIONAL FLAGS (AGGREGATE ACROSS SEEDS BEFORE CLAIMING)")
print("=" * 100)
print(directional_df.to_string(index=False))


# -----------------------------------------------------------------------------
# 9. Machine-readable output
# Copy each delimited block directly into a correspondingly named .csv file.
# The JSON bundle contains the same tables as lists of records.
# -----------------------------------------------------------------------------
e1_tables = {
    "geometry": geometry_df,
    "coefficients": coefficient_df,
    "nll_increment": nll_df,
    "residuals": residual_df,
    "runtime": runtime_df,
    "primary": primary_df,
    "pairwise": pairwise_df,
    "directional_flags": directional_df,
}

for table_name, table_df in e1_tables.items():
    print(f"\n===== BEGIN E1_{table_name.upper()}_CSV =====")
    print(table_df.to_csv(index=False), end="")
    print(f"===== END E1_{table_name.upper()}_CSV =====")

e1_result_bundle = {
    "manifest": run_manifest,
    "tables": {
        table_name: table_df.to_dict(orient="records")
        for table_name, table_df in e1_tables.items()
    },
}

print("\n===== BEGIN E1_RESULTS_JSON =====")
print(
    json.dumps(
        e1_json_ready(e1_result_bundle),
        indent=2,
        sort_keys=True,
    )
)
print("===== END E1_RESULTS_JSON =====")
